<a href="https://colab.research.google.com/github/Oyakata32/Lesson-16/blob/main/Lekcja_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import math
# Parametry rozkładu normalnego
mean = 170 # średnia (μ) - wzrost w cm
std = 10 # odchylenie standardowe (σ)
# Generowanie danych z rozkładu normalnego
rng = np.random.default_rng(seed=42)
heights = rng.normal(loc=mean, scale=std, size=1000)
# --- Ręczne implementacje PDF i CDF ---
def normal_pdf(x, mu, sigma):
  """Gęstość rozkładu normalnego: f(x) = 1/(σ√(2π)) * exp(-0.5 * ((x-μ)/σ)²)."""
  return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))
# Dystrybuanta CDF korzysta z funkcji błędu erf (ze stdlib math).
# Dla skalara: 0.5*(1+erf((x-μ)/(σ√2))). Wektoryzujemy przez np.vectorize.
_erf_vec = np.vectorize(math.erf)
def normal_cdf(x, mu, sigma):
  """Dystrybuanta rozkładu normalnego przez math.erf."""
  return 0.5 * (1.0 + _erf_vec((x - mu) / (sigma * np.sqrt(2))))
# --- ASCII histogram (bez matplotlib) ---
def ascii_histogram(data, bins=30, height=10, label="Histogram"):
  counts, edges = np.histogram(data, bins=bins)
  max_count = counts.max() if counts.max() > 0 else 1
  scaled = np.round(counts / max_count * height).astype(int)
  print(f"\n{label} (max={max_count}):")
  for h in range(height, 0, -1):
    row = "".join("█" if s >= h else " " for s in scaled)
    print(f" {row}")
    print(" " + "─" * bins)
    print(f" {edges[0]:>5.1f}{' ' * (bins - 12)}{edges[-1]:>5.1f}")
# Empiryczny histogram próbki
ascii_histogram(heights, bins=30, height=10, label="Empiryczny rozkład wzrostu N(170, 10)")
# Teoretyczna krzywa PDF - tabela wartości
x_grid = np.linspace(mean - 4 * std, mean + 4 * std, 17)
pdf_vals = normal_pdf(x_grid, mean, std)
print("\nTeoretyczna PDF rozkładu normalnego N(170, 10):")
for xi, pi in zip(x_grid, pdf_vals):
  bar_len = int(pi / pdf_vals.max() * 40)
  print(f" x={xi:6.1f} f(x)={pi:.5f} {'█' * bar_len}")
# --- Obliczanie prawdopodobieństw przez ręczną CDF ---
prob_below_160 = float(normal_cdf(160, mean, std))
print(f"\nP(wzrost < 160cm) = {prob_below_160:.4f} ({prob_below_160*100:.2f}%)")
prob_between = float(normal_cdf(180, mean, std) - normal_cdf(160, mean, std))
print(f"P(160 < wzrost < 180) = {prob_between:.4f} ({prob_between*100:.2f}%)")
# Reguła trzech sigm
prob_1sigma = float(normal_cdf(mean + std, mean, std) - normal_cdf(mean - std, mean, std))
prob_2sigma = float(normal_cdf(mean + 2*std, mean, std) - normal_cdf(mean - 2*std, mean,
std))
prob_3sigma = float(normal_cdf(mean + 3*std, mean, std) - normal_cdf(mean - 3*std, mean,
std))
print(f"\nReguła trzech sigm:")
print(f" 68.27% danych w [μ-σ, μ+σ]: {prob_1sigma:.4f}")
print(f" 95.45% danych w [μ-2σ, μ+2σ]: {prob_2sigma:.4f}")
print(f" 99.73% danych w [μ-3σ, μ+3σ]: {prob_3sigma:.4f}")
# Weryfikacja empiryczna
empirical_1sigma = np.mean(np.abs(heights - mean) <= std)
empirical_2sigma = np.mean(np.abs(heights - mean) <= 2 * std)
empirical_3sigma = np.mean(np.abs(heights - mean) <= 3 * std)
print(f"\nEmpiryczne frakcje (n=1000):")
print(f" w [μ-σ, μ+σ]: {empirical_1sigma:.4f}")
print(f" w [μ-2σ, μ+2σ]: {empirical_2sigma:.4f}")
print(f" w [μ-3σ, μ+3σ]: {empirical_3sigma:.4f}")


Empiryczny rozkład wzrostu N(170, 10) (max=119):
                  █            
 ──────────────────────────────
 133.5                  201.8
                  █            
 ──────────────────────────────
 133.5                  201.8
               █  █            
 ──────────────────────────────
 133.5                  201.8
               ████            
 ──────────────────────────────
 133.5                  201.8
               █████           
 ──────────────────────────────
 133.5                  201.8
            █████████          
 ──────────────────────────────
 133.5                  201.8
            ██████████         
 ──────────────────────────────
 133.5                  201.8
          █████████████        
 ──────────────────────────────
 133.5                  201.8
         ████████████████      
 ──────────────────────────────
 133.5                  201.8
      █████████████████████    
 ──────────────────────────────
 133.5                  201.8

Teoretycz

In [ ]:
import numpy as np
import math
_erf_vec = np.vectorize(math.erf)
def make_synthetic_flowers(seed=42):
  """Kanoniczny generator z L15: 150 × 4 cechy, 3 gatunki A/B/C."""
  rng = np.random.default_rng(seed)
  n_per = 50
  # 4 cechy: sepal_length, sepal_width, petal_length, petal_width
  means = np.array([
  [5.0, 3.4, 1.5, 0.25], # A (niskie płatki)
  [5.9, 2.8, 4.3, 1.30], # B (średnie)
  [6.6, 3.0, 5.6, 2.00], # C (wysokie)
  ])
  std = np.array([0.35, 0.35, 0.45, 0.30])
  X_parts = [rng.normal(loc=means[i], scale=std, size=(n_per, 4)) for i in range(3)]
  X = np.vstack(X_parts)
  y = np.repeat(["A", "B", "C"], n_per)
  return X, y
X, species = make_synthetic_flowers(seed=42)
feature_names = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
# Podstawowe statystyki
print(f"Shape danych: {X.shape}")
print(f"{'cecha':<15} {'mean':>8} {'std':>8} {'min':>8} {'max':>8}")
for i, name in enumerate(feature_names):
  col = X[:, i]
  print(f"{name:<15} {col.mean():>8.3f} {col.std():>8.3f} {col.min():>8.3f}{col.max():>8.3f}")
# --- Ręczny test normalności: skewness + kurtosis ---
# Skewness: 0 = symetria, >0 = prawostronna, <0 = lewostronna
# Kurtosis (excess): 0 = jak normalny, >0 = cięższe ogony
def skewness(x):
  x = np.asarray(x, dtype=float)
  m = x.mean()
  s = x.std(ddof=0)
  return ((x - m) ** 3).mean() / (s ** 3)
def kurtosis_excess(x):
  x = np.asarray(x, dtype=float)
  m = x.mean()
  s = x.std(ddof=0)
  return ((x - m) ** 4).mean() / (s ** 4) - 3.0
# Test Jarque-Bera (manualny): JB = n/6 * (skew² + kurt²/4)
# H0: dane pochodzą z rozkładu normalnego
# Przy H0 statystyka JB ma rozkład chi² o 2 stopniach swobody
# p-value ≈ exp(-JB/2) dla chi²(2) (dokładnie: CDF = 1 - exp(-JB/2))
def jarque_bera_manual(x):
  n = len(x)
  sk = skewness(x)
  ku = kurtosis_excess(x)
  jb = n / 6.0 * (sk ** 2 + (ku ** 2) / 4.0)
  # Chi-square(2) has CDF F(x) = 1 - exp(-x/2). p-value = 1 - F(jb) = exp(-jb/2).
  p_value = math.exp(-jb / 2)
  return jb, p_value, sk, ku
print("\n" + "=" * 60)
print("TEST NORMALNOŚCI (Jarque-Bera - ręczna implementacja)")
print("=" * 60)
for i, name in enumerate(feature_names):
  col = X[:, i]
  jb, p_value, sk, ku = jarque_bera_manual(col)
  verdict = "JEST normalny" if p_value > 0.05 else "NIE JEST normalny"
  print(f"\n{name}:")
  print(f" JB statystyka = {jb:.4f}, p-value = {p_value:.4f}")
  print(f" Skewness: {sk:+.4f} (0=symetria)")
  print(f" Kurtosis (excess): {ku:+.4f} (0=jak normalny)")
  print(f" → Rozkład {verdict}")
# --- ASCII Q-Q plot ---
# Sortujemy dane, porównujemy z kwantylami teoretycznego N(0,1)
# Odwrotna dystrybuanta Phi^(-1)(p) aproksymowana wzorem Beasley-Springera-Moro,
# ale dla przejrzystości użyjemy prostej aproksymacji przez bisekcję.
def normal_cdf(z):
  return 0.5 * (1 + _erf_vec(z / np.sqrt(2)))
def normal_ppf(p, iterations=60):
  """Odwrotna CDF N(0,1) przez bisekcję (wystarczająco dokładna do wizualizacji)."""
  p = np.asarray(p)
  lo = np.full_like(p, -8.0, dtype=float)
  hi = np.full_like(p, 8.0, dtype=float)
  for _ in range(iterations):
    mid = (lo + hi) / 2
    cdf = normal_cdf(mid)
    lo = np.where(cdf < p, mid, lo)
    hi = np.where(cdf >= p, mid, hi)
  return (lo + hi) / 2
def ascii_qq_plot(data, name, rows=12, cols=40):
  """ASCII Q-Q plot: kwantyle empiryczne vs teoretyczne N(0,1)."""
  n = len(data)
  sorted_data = np.sort(data)
  # Standaryzacja: z = (x-mean)/std
  z_empirical = (sorted_data - sorted_data.mean()) / sorted_data.std(ddof=0)
  # Teoretyczne kwantyle: p_i = (i - 0.5)/n
  p = (np.arange(1, n + 1) - 0.5) / n
  z_theoretical = normal_ppf(p)
  # Rysujemy siatkę rows×cols
  x_min, x_max = z_theoretical.min(), z_theoretical.max()
  y_min, y_max = z_empirical.min(), z_empirical.max()
  y_range = max(y_max - y_min, 1e-9)
  x_range = max(x_max - x_min, 1e-9)
  grid = [[" "] * cols for _ in range(rows)]
  for xt, ye in zip(z_theoretical, z_empirical):
    col = int((xt - x_min) / x_range * (cols - 1))
    row = rows - 1 - int((ye - y_min) / y_range * (rows - 1))
    grid[row][col] = "*"
    # Linia idealna y=x: rysujemy kropkami
  for c in range(cols):
    xt = x_min + c / (cols - 1) * x_range
    row = rows - 1 - int((xt - y_min) / y_range * (rows - 1))
  if 0 <= row < rows and grid[row][c] == " ":
    grid[row][c] = "·"
    print(f"\nQ-Q plot: {name} (* = dane, · = idealna linia y=x)")
  for row in grid:
    print(" |" + "".join(row) + "|")
  print(" " + "─" * (cols + 2))
  print(f" kwantyle teoretyczne N(0,1): {x_min:+.2f} .. {x_max:+.2f}")
for i, name in enumerate(feature_names):
  ascii_qq_plot(X[:, i], name)

Shape danych: (150, 4)
cecha               mean      std      min      max
sepal_length       5.820    0.752    4.317   7.283
sepal_width        3.053    0.418    1.902   3.978
petal_length       3.797    1.727    0.541   6.689
petal_width        1.177    0.764   -0.184   2.779

TEST NORMALNOŚCI (Jarque-Bera - ręczna implementacja)

sepal_length:
 JB statystyka = 7.5351, p-value = 0.0231
 Skewness: -0.0666 (0=symetria)
 Kurtosis (excess): -1.0899 (0=jak normalny)
 → Rozkład NIE JEST normalny

sepal_width:
 JB statystyka = 1.0293, p-value = 0.5977
 Skewness: -0.1215 (0=symetria)
 Kurtosis (excess): -0.3250 (0=jak normalny)
 → Rozkład JEST normalny

petal_length:
 JB statystyka = 15.0040, p-value = 0.0006
 Skewness: -0.3997 (0=symetria)
 Kurtosis (excess): -1.3272 (0=jak normalny)
 → Rozkład NIE JEST normalny

petal_width:
 JB statystyka = 8.7114, p-value = 0.0128
 Skewness: -0.1603 (0=symetria)
 Kurtosis (excess): -1.1363 (0=jak normalny)
 → Rozkład NIE JEST normalny
 |                 

In [ ]:
import math
import numpy as np
# --- Ręczna implementacja PMF ---
def binom_pmf(k, n, p):
  """PMF rozkładu dwumianowego: C(n,k) * p^k * (1-p)^(n-k)."""
  return math.comb(n, k) * (p ** k) * ((1 - p) ** (n - k))
def poisson_pmf(k, lam):
  """PMF rozkładu Poissona: λ^k * exp(-λ) / k!."""
  return (lam ** k) * math.exp(-lam) / math.factorial(k)
# --- ROZKŁAD BINOMIALNY ---
# Przykład: rzut monetą 10 razy, prawdopodobieństwo p=0.5
n_trials = 10
p_success = 0.5
x_binom = np.arange(0, n_trials + 1)
pmf_binom = np.array([binom_pmf(int(k), n_trials, p_success) for k in x_binom])
# --- ROZKŁAD POISSONA ---
# Przykład: liczba klientów w sklepie na godzinę, λ=3
lambda_rate = 3
x_poisson = np.arange(0, 15)
pmf_poisson = np.array([poisson_pmf(int(k), lambda_rate) for k in x_poisson])
# --- ASCII bar chart dla PMF ---
def ascii_barplot_pmf(xs, ps, title, bar_char="█", max_width=40):
  print(f"\n{title}")
  p_max = ps.max() if ps.max() > 0 else 1
  for xi, pi in zip(xs, ps):
    bar_len = int(pi / p_max * max_width)
    print(f" x={int(xi):>3} p={pi:.5f} {bar_char * bar_len}")
ascii_barplot_pmf(x_binom, pmf_binom,
f"Rozkład Binomialny B(n={n_trials}, p={p_success})")
ascii_barplot_pmf(x_poisson, pmf_poisson,
f"Rozkład Poissona (λ={lambda_rate})")
# --- Wartości oczekiwane i odchylenia (z wzorów) ---
# Binomialny: E(X) = np, Var(X) = np(1-p)
binom_mean = n_trials * p_success
binom_std = np.sqrt(n_trials * p_success * (1 - p_success))
# Poisson: E(X) = Var(X) = λ
poisson_mean = lambda_rate
poisson_std = np.sqrt(lambda_rate)
# --- Obliczenia prawdopodobieństw ---
print("\nROZKŁAD BINOMIALNY (10 rzutów monetą):")
print(f" P(X = 5 orłów) = {binom_pmf(5, n_trials, p_success):.4f}")
# P(X >= 7) = sum_{k=7..10} pmf(k)
p_ge_7 = sum(binom_pmf(k, n_trials, p_success) for k in range(7, n_trials + 1))
print(f" P(X >= 7 orłów) = {p_ge_7:.4f}")
print(f" Wartość oczekiwana E(X) = np = {binom_mean:.2f}")
print(f" Odchylenie standardowe σ = √(np(1-p)) = {binom_std:.2f}")
print("\nROZKŁAD POISSONA (klienci w sklepie, λ=3):")
print(f" P(X = 2 klientów) = {poisson_pmf(2, lambda_rate):.4f}")
p_le_5 = sum(poisson_pmf(k, lambda_rate) for k in range(0, 6))
p_gt_5 = 1 - p_le_5
print(f" P(X <= 5 klientów) = {p_le_5:.4f}")
print(f" P(X > 5 klientów) = {p_gt_5:.4f}")
print(f" Wartość oczekiwana E(X) = λ = {poisson_mean:.2f}")
print(f" Odchylenie standardowe σ = √λ = {poisson_std:.2f}")


Rozkład Binomialny B(n=10, p=0.5)
 x=  0 p=0.00098 
 x=  1 p=0.00977 █
 x=  2 p=0.04395 ███████
 x=  3 p=0.11719 ███████████████████
 x=  4 p=0.20508 █████████████████████████████████
 x=  5 p=0.24609 ████████████████████████████████████████
 x=  6 p=0.20508 █████████████████████████████████
 x=  7 p=0.11719 ███████████████████
 x=  8 p=0.04395 ███████
 x=  9 p=0.00977 █
 x= 10 p=0.00098 

Rozkład Poissona (λ=3)
 x=  0 p=0.04979 ████████
 x=  1 p=0.14936 ██████████████████████████
 x=  2 p=0.22404 ████████████████████████████████████████
 x=  3 p=0.22404 ████████████████████████████████████████
 x=  4 p=0.16803 ██████████████████████████████
 x=  5 p=0.10082 ██████████████████
 x=  6 p=0.05041 █████████
 x=  7 p=0.02160 ███
 x=  8 p=0.00810 █
 x=  9 p=0.00270 
 x= 10 p=0.00081 
 x= 11 p=0.00022 
 x= 12 p=0.00006 
 x= 13 p=0.00001 
 x= 14 p=0.00000 

ROZKŁAD BINOMIALNY (10 rzutów monetą):
 P(X = 5 orłów) = 0.2461
 P(X >= 7 orłów) = 0.1719
 Wartość oczekiwana E(X) = np = 5.00
 Odchyleni

In [ ]:
import numpy as np
import pandas as pd
import math
_erf_vec = np.vectorize(math.erf)
def make_california_housing(n=2000, seed=42):
  """Kanoniczny generator syntetycznych danych California Housing (z L12)."""
  rng = np.random.default_rng(seed)
  MedInc = rng.gamma(shape=2.5, scale=1.6, size=n) + 0.5
  HouseAge = rng.uniform(1, 52, size=n)
  AveRooms = 3 + 0.8 * MedInc + rng.normal(0, 0.8, size=n)
  AveBedrms = 1 + 0.1 * AveRooms + rng.normal(0, 0.1, size=n)
  Population = rng.gamma(shape=2.0, scale=700, size=n)
  AveOccup = 2.5 + rng.normal(0, 0.6, size=n)
  Latitude = rng.uniform(32.5, 42.0, size=n)
  Longitude = rng.uniform(-124.5, -114.0, size=n)
  MedHouseVal = (
  0.45 * MedInc
  - 0.01 * HouseAge
  + 0.03 * AveRooms
  - 0.02 * AveBedrms
  - 0.0001 * Population
  + 0.12 * (Latitude - 36)
  - 0.05 * (Longitude + 119)
  + rng.normal(0, 0.4, size=n)
  )
  MedHouseVal = np.clip(MedHouseVal, 0.15, 5.0)
  return pd.DataFrame({
  "MedInc": MedInc, "HouseAge": HouseAge, "AveRooms": AveRooms,
  "AveBedrms": AveBedrms, "Population": Population, "AveOccup": AveOccup,
  "Latitude": Latitude, "Longitude": Longitude, "MedHouseVal": MedHouseVal,
  })
housing = make_california_housing(n=2000, seed=42)
print("Syntetyczny dataset California Housing (z L12):")
print(housing.head())
print(f"\nShape: {housing.shape}")
print(f"Kolumny: {list(housing.columns)}")
# Podstawowe statystyki (pandas .describe z L11)
print("\nPodstawowe statystyki:")
print(housing.describe().round(3))
# --- Macierz korelacji przez np.corrcoef ---
# np.corrcoef(X.T) zwraca macierz Pearson korelacji dla kolumn macierzy X.
col_names = list(housing.columns)
corr_matrix = np.corrcoef(housing.values.T)
# Korelacja z MedHouseVal
target_idx = col_names.index("MedHouseVal")
target_corr = corr_matrix[target_idx]
print("\nKorelacja z MedHouseVal (sortowana):")
order = np.argsort(-target_corr)
for i in order:
  print(f" {col_names[i]:<15} r = {target_corr[i]:+.4f}")
# --- ASCII heatmap macierzy korelacji ---
def corr_to_char(r):
  """Mapuje r ∈ [-1, 1] na znak: negatywne jasno, pozytywne ciemno."""
  if r >= 0.75: return "█"
  if r >= 0.50: return "▓"
  if r >= 0.25: return "▒"
  if r >= 0.05: return "░"
  if r > -0.05: return "·"
  if r > -0.25: return "-"
  if r > -0.50: return "="
  if r > -0.75: return "#"
  return "@"
def ascii_heatmap(M, labels):
  n = len(labels)
  # Skróć etykiety do 7 znaków
  short = [l[:7] for l in labels]
  print("\nASCII heatmap macierzy korelacji (legenda: █≥.75 ▓≥.5 ▒≥.25 ░≥.05 ·≈0 -/=≤0#/@≤-.5):")
  header = " " * 9 + "".join(f"{s:>7}" for s in short)
  print(header)
  for i in range(n):
    row = f"{short[i]:>8} "
    for j in range(n):
      row += f" {corr_to_char(M[i,j])} {M[i,j]:+.2f}"[:7]
    print(row)
ascii_heatmap(corr_matrix, col_names)
# --- Szczegółowa analiza: MedInc vs MedHouseVal ---
feature_x = "MedInc"
feature_y = "MedHouseVal"
x = housing[feature_x].values
y = housing[feature_y].values
# Współczynnik Pearsona ręcznie: r = cov(x,y) / (std(x) * std(y))
r = np.corrcoef(x, y)[0, 1]
# p-value dla Pearson r: t = r * sqrt((n-2)/(1-r²)) ma rozkład t-Studenta o df=n-2
# Dla dużego n (>30) t → N(0,1), więc p-value ≈ 2 * (1 - Φ(|t|))
n = len(x)
t_stat = r * np.sqrt((n - 2) / max(1 - r ** 2, 1e-12))
# p-value przez ręczną CDF rozkładu normalnego (przybliżenie dla dużych n)
p_value = 2 * (1 - 0.5 * (1 + _erf_vec(abs(t_stat) / np.sqrt(2))))
print(f"\nKorelacja Pearsona między {feature_x} a {feature_y}:")
print(f" Współczynnik korelacji r = {r:.4f}")
print(f" t-statystyka = {t_stat:.4f} (df={n-2})")
print(f" p-value ≈ {float(p_value):.2e} (aproksymacja normalna dla dużego n)")
# Interpretacja
if abs(r) > 0.7:
  strength = "silna"
elif abs(r) > 0.4:
  strength = "umiarkowana"
elif abs(r) > 0.2:
  strength = "słaba"
else:
  strength = "bardzo słaba"
direction = "dodatnia" if r > 0 else "ujemna"
print(f" Interpretacja: Korelacja {strength} {direction}")
# --- ASCII scatter plot z linią regresji ---
def ascii_scatter(x, y, title, rows=14, cols=50):
  x_min, x_max = x.min(), x.max()
  y_min, y_max = y.min(), y.max()
  x_range = max(x_max - x_min, 1e-9)
  y_range = max(y_max - y_min, 1e-9)
  grid = [[" "] * cols for _ in range(rows)]
  for xi, yi in zip(x, y):
    c = int((xi - x_min) / x_range * (cols - 1))
    r_ = rows - 1 - int((yi - y_min) / y_range * (rows - 1))
    if grid[r_][c] == " ":
      grid[r_][c] = "·"
    elif grid[r_][c] == "·":
      grid[r_][c] = "*"
    else:
      grid[r_][c] = "#" # gęsty punkt
  # Linia regresji
  slope, intercept = np.polyfit(x, y, 1)
  for c in range(cols):
    xi = x_min + c / (cols - 1) * x_range
    yi = slope * xi + intercept
    r_ = rows - 1 - int((yi - y_min) / y_range * (rows - 1))
    if 0 <= r_ < rows and grid[r_][c] in (" ", "·"):
      grid[r_][c] = "─"
  print(f"\n{title}")
  print(f"y = {slope:.3f}·x + {intercept:.3f}")
  for row in grid:
    print(" |" + "".join(row) + "|")
  print(" " + "─" * (cols + 2))
  print(f" x: [{x_min:.2f}, {x_max:.2f}] y: [{y_min:.2f}, {y_max:.2f}]")
ascii_scatter(x, y, f"Scatter: {feature_x} vs {feature_y} r = {r:.3f}")

Syntetyczny dataset California Housing (z L12):
     MedInc   HouseAge  AveRooms  AveBedrms   Population  AveOccup   Latitude  \
0  4.734977  40.311324  6.884812   1.606462   885.607473  1.436806  40.967157   
1  6.051457  26.516096  9.062208   2.065442  2715.582991  2.251636  38.671920   
2  4.276549  40.160726  6.771253   1.684698  1021.146138  3.267557  38.638543   
3  3.927248  14.661656  6.282833   1.674368  1037.793205  2.655178  40.260311   
4  6.477596  31.064823  8.022173   1.775457  1053.317499  1.649014  34.068856   

    Longitude  MedHouseVal  
0 -114.373283     1.862964  
1 -120.034416     3.384592  
2 -124.278583     1.891606  
3 -118.824055     2.341919  
4 -117.400009     2.347516  

Shape: (2000, 9)
Kolumny: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'MedHouseVal']

Podstawowe statystyki:
         MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  \
count  2000.000  2000.000  2000.000   2000.000    2000

In [ ]:
import numpy as np
import math
_erf_vec = np.vectorize(math.erf)
def ranks(x):
  """Rangi elementów wektora x (1..n). Ties są ignorowane - używamy ordinal ranking."""
  return np.argsort(np.argsort(x)) + 1
def pearson_corr(x, y):
  """Ręczny Pearson: r = cov(x,y) / (std(x) * std(y))."""
  x = np.asarray(x, dtype=float)
  y = np.asarray(y, dtype=float)
  xm, ym = x.mean(), y.mean()
  cov = ((x - xm) * (y - ym)).mean()
  return cov / (x.std(ddof=0) * y.std(ddof=0))
def spearman_corr(x, y):
  """Spearman = Pearson na rangach."""
  return pearson_corr(ranks(x), ranks(y))
def corr_pvalue_approx(r, n):
  """Aproksymacja normalna p-value dla Pearson r lub Spearman ρ."""
  t = r * np.sqrt((n - 2) / max(1 - r ** 2, 1e-12))
  return 2 * (1 - 0.5 * (1 + float(_erf_vec(abs(t) / np.sqrt(2)))))
# Tworzenie danych z różnymi typami zależności
rng = np.random.default_rng(seed=42)
n = 100
# 1. Zależność liniowa (Pearson i Spearman wykryją)
x_linear = np.linspace(0, 10, n)
y_linear = 2 * x_linear + rng.normal(0, 1, n)
# 2. Zależność nieliniowa monotoniczna (Spearman wykryje lepiej)
x_nonlinear = np.linspace(0, 10, n)
y_nonlinear = x_nonlinear ** 2 + rng.normal(0, 5, n)
# 3. Zależność z outlierami (Spearman odporny)
x_outliers = np.linspace(0, 10, n)
y_outliers = 2 * x_outliers + rng.normal(0, 1, n)
y_outliers[0] = 50
y_outliers[1] = -30
datasets = [
("Liniowa", x_linear, y_linear),
("Nieliniowa (kwadratowa)", x_nonlinear, y_nonlinear),
("Z outlierami", x_outliers, y_outliers),
]
def ascii_scatter_mini(x, y, rows=10, cols=36):
  x_min, x_max = x.min(), x.max()
  y_min, y_max = y.min(), y.max()
  x_range = max(x_max - x_min, 1e-9)
  y_range = max(y_max - y_min, 1e-9)
  grid = [[" "] * cols for _ in range(rows)]
  for xi, yi in zip(x, y):
    c = int((xi - x_min) / x_range * (cols - 1))
    r_ = rows - 1 - int((yi - y_min) / y_range * (rows - 1))
    grid[r_][c] = "*"
  for row in grid:
    print(" |" + "".join(row) + "|")
for name, x, y in datasets:
  pr = pearson_corr(x, y)
  sr = spearman_corr(x, y)
  pr_p = corr_pvalue_approx(pr, len(x))
  sr_p = corr_pvalue_approx(sr, len(x))
  print(f"\n{name}:")
  print(f" Pearson r = {pr:+.4f} (p≈{pr_p:.4f})")
  print(f" Spearman ρ = {sr:+.4f} (p≈{sr_p:.4f})")
  ascii_scatter_mini(x, y)


Liniowa:
 Pearson r = +0.9912 (p≈0.0000)
 Spearman ρ = +0.9920 (p≈0.0000)
 |                                  * |
 |                               *****|
 |                          *****  *  |
 |                      * ***         |
 |                  **** **           |
 |             ******* *              |
 |         ** ***                     |
 |    ********                        |
 | *******                            |
 |****                                |

Nieliniowa (kwadratowa):
 Pearson r = +0.9556 (p≈0.0000)
 Spearman ρ = +0.9736 (p≈0.0000)
 |                                   *|
 |                               **** |
 |                             ** **  |
 |                          ******    |
 |                         ***        |
 |                     ******         |
 |             *  *  ****             |
 |             ******                 |
 |* *   ******* **                    |
 |******** **                         |

Z outlierami:
 Pearson r = +0.682

In [ ]:
import numpy as np
import pandas as pd
def make_synthetic_medical(n_samples=500, seed=42):
  """
  Syntetyczny dataset medyczny z 30 cechami. Pary i trójki cech są silnie skorelowane
  (np. mean_radius/mean_area/mean_perimeter), co symuluje problem multikolinearności.
  """
  rng = np.random.default_rng(seed)
  # 10 "ukrytych" czynników - to prawdziwe niezależne zmienne
  latent = rng.normal(size=(n_samples, 10))
  # Każda cecha = kombinacja liniowa kilku czynników + szum
  feature_names = [
  "mean_radius", "mean_area", "mean_perimeter", # silnie skorelowane (1 latent)
  "mean_smoothness", "mean_compactness",
  "mean_concavity", "mean_concave_pts", # skorelowane z compactness
  "mean_symmetry", "mean_fractal",
  "se_radius", "se_area", "se_perimeter", # SE też skorelowane (jak mean_*)
  "se_smoothness", "se_compactness",
  "se_concavity", "se_concave_pts",
  "se_symmetry", "se_fractal",
  "worst_radius", "worst_area", "worst_perimeter",# worst też skorelowane
  "worst_smoothness", "worst_compactness",
  "worst_concavity", "worst_concave_pts",
  "worst_symmetry", "worst_fractal",
  "tissue_density", "patient_age", # mniej skorelowane
  ]
  # Macierz wag: (30 cech, 10 latent) - sterujemy strukturą korelacji
  W = np.zeros((29, 10))
  # Klastry skorelowanych cech
  W[0:3, 0] = [1.0, 1.0, 0.95] # mean size
  W[3:5, 1] = [1.0, 0.9] # mean texture
  W[5:7, 1] = [0.85, 0.85] # concavity silnie skor. z compactness
  W[7:9, 2] = [1.0, 0.8] # symmetry/fractal
  W[9:12, 0] = [0.9, 0.9, 0.85] # SE size - silnie skor. z mean size!
  W[12:14, 3] = [1.0, 0.9]
  W[14:16, 3] = [0.85, 0.85]
  W[16:18, 4] = [1.0, 0.8]
  W[18:21, 0] = [0.95, 0.95, 0.90] # worst size - skor. z mean size!
  W[21:23, 5] = [1.0, 0.9]
  W[23:25, 5] = [0.85, 0.85]
  W[25:27, 6] = [1.0, 0.8]
  W[27, 7] = 1.0 # tissue_density
  W[28, 8] = 1.0 # patient_age
  # Szum uzupełnia
  X = latent @ W.T + rng.normal(scale=0.3, size=(n_samples, 29))
  # Target: kombinacja latent[0] (rozmiar) + latent[5]
  target = ((latent[:, 0] * 0.6 + latent[:, 5] * 0.4) > 0).astype(int)
  # Skalowanie do dodatnich (jak w rzeczywistych danych medycznych)
  X = X + 10
  df = pd.DataFrame(X, columns=feature_names)
  df["target"] = target
  return df
cancer = make_synthetic_medical(n_samples=500, seed=42)
print(f"Dataset syntetyczny medyczny: {cancer.shape}")
print(f"Liczba cech: {len(cancer.columns) - 1}")
# --- Macierz korelacji ---
feature_cols = [c for c in cancer.columns if c != "target"]
corr_matrix = np.corrcoef(cancer[feature_cols].values.T)
# --- Znalezienie par cech z wysoką korelacją ---
threshold = 0.9
high_corr_pairs = []
n_features = len(feature_cols)
for i in range(n_features):
  for j in range(i + 1, n_features):
    r = corr_matrix[i, j]
    if abs(r) > threshold:
      high_corr_pairs.append((feature_cols[i], feature_cols[j], r))
high_corr_pairs.sort(key=lambda t: -abs(t[2]))
print(f"\nPary cech z |korelacją| > {threshold}: {len(high_corr_pairs)}")
print(f"\n{'cecha 1':<20} {'cecha 2':<20} {'r':>8}")
for a, b, r in high_corr_pairs[:15]:
  print(f"{a:<20} {b:<20} {r:+.4f}")
# --- Top 10 cech skorelowanych z targetem ---
# Korelacja z targetem (punkt-biserialna, obliczana jak Pearson)
X_arr = cancer[feature_cols].values
y_arr = cancer["target"].values.astype(float)
target_corr = np.array([np.corrcoef(X_arr[:, k], y_arr)[0, 1] for k in range(n_features)])
top_idx = np.argsort(-np.abs(target_corr))[:10]
top_features = [feature_cols[i] for i in top_idx]
print("\nTop 10 cech skorelowanych z targetem:")
for i in top_idx:
  print(f" {feature_cols[i]:<20} r = {target_corr[i]:+.4f}")
# --- ASCII heatmap top 10 cech ---
top_corr = np.corrcoef(X_arr[:, top_idx].T)
def corr_char(r):
  if r >= 0.75: return "█"
  if r >= 0.50: return "▓"
  if r >= 0.25: return "▒"
  if r >= 0.05: return "░"
  if r > -0.05: return "·"
  if r > -0.25: return "-"
  if r > -0.50: return "="
  if r > -0.75: return "#"
  return "@"
print("\nASCII heatmap top 10 cech (legenda: █≥.75 ▓≥.5 ▒≥.25 ░≥.05 ·≈0 -/=<0 #/@<-.5):")
short = [c[:7] for c in top_features]
print(" " * 9 + "".join(f"{s:>8}" for s in short))
for i in range(10):
  row = f"{short[i]:>8} "
  for j in range(10):
    row += f" {corr_char(top_corr[i, j])}{top_corr[i, j]:+.2f}"
  print(row)
# --- Feature selection: usuwanie multikolinearnych cech ---
def remove_multicollinear(corr_abs, feat_names, threshold=0.95):
  """
  Iteracyjnie usuwa cechę z górnego trójkąta macierzy korelacji,
  jeśli jakakolwiek jej wartość > threshold.
  """
  to_drop = []
  keep = list(range(len(feat_names)))
  M = corr_abs.copy()
  # Upper triangle scan
  for i in range(len(feat_names)):
    for j in range(i + 1, len(feat_names)):
      if i in keep and j in keep and M[i, j] > threshold:
        # Usuwamy drugą (j) - zostawiamy pierwszą
        to_drop.append(feat_names[j])
        keep.remove(j)
  return to_drop
features_to_drop = remove_multicollinear(np.abs(corr_matrix), feature_cols, threshold=0.95)
print(f"\nCechy do usunięcia (|r| > 0.95): {len(features_to_drop)}")
print(features_to_drop[:10])
cancer_reduced = cancer.drop(columns=features_to_drop)
print(f"\nPoczątkowa liczba cech: {n_features}")
print(f"Po usunięciu multikolinearności: {len(cancer_reduced.columns) - 1}")

Dataset syntetyczny medyczny: (500, 30)
Liczba cech: 29

Pary cech z |korelacją| > 0.9: 32

cecha 1              cecha 2                     r
worst_area           worst_perimeter      +0.9192
worst_smoothness     worst_compactness    +0.9187
mean_radius          worst_area           +0.9173
se_smoothness        se_compactness       +0.9168
mean_radius          mean_area            +0.9165
mean_radius          se_radius            +0.9145
mean_radius          worst_perimeter      +0.9140
se_radius            worst_perimeter      +0.9132
mean_radius          mean_perimeter       +0.9127
worst_symmetry       worst_fractal        +0.9120
se_smoothness        se_concavity         +0.9119
mean_smoothness      mean_compactness     +0.9112
mean_radius          se_area              +0.9108
mean_perimeter       worst_area           +0.9103
mean_area            worst_area           +0.9097

Top 10 cech skorelowanych z targetem:
 worst_area           r = +0.6595
 se_radius            r = +0.6584


In [ ]:
import numpy as np
import math
_erf_vec = np.vectorize(math.erf)
def make_synthetic_flowers(seed=42):
  """Kanoniczny generator z L15: 150 × 4 cechy, 3 gatunki A/B/C."""
  rng = np.random.default_rng(seed)
  n_per = 50
  means = np.array([
  [5.0, 3.4, 1.5, 0.25],
  [5.9, 2.8, 4.3, 1.30],
  [6.6, 3.0, 5.6, 2.00],
  ])
  std = np.array([0.35, 0.35, 0.45, 0.30])
  X_parts = [rng.normal(loc=means[i], scale=std, size=(n_per, 4)) for i in range(3)]
  return np.vstack(X_parts), np.repeat(["A", "B", "C"], n_per)
X, species = make_synthetic_flowers(seed=42)
# Dla kompatybilności z notacją: petal_length to kolumna 2
petal_length = X[:, 2]
# --- PRZYPADEK 1: Test t jednej próbki ---
# H₀: średnia długość płatka = 4.0 cm
# H₁: średnia długość płatka ≠ 4.0 cm
sample_mean = petal_length.mean()
hypothesized_mean = 4.0
n = len(petal_length)

s = petal_length.std(ddof=1) # sample std z poprawką Bessela (ddof=1)
# t = (x̄ - μ₀) / (s / √n)
t_stat_1samp = (sample_mean - hypothesized_mean) / (s / np.sqrt(n))
# Aproksymacja normalna p-value (dla n > 30 t ≈ N(0,1))
def two_sided_pvalue_normal(t):
  return 2 * (1 - 0.5 * (1 + float(_erf_vec(abs(t) / np.sqrt(2)))))
p_value_1samp = two_sided_pvalue_normal(t_stat_1samp)
print("TEST T JEDNEJ PRÓBKI")
print("=" * 60)
print(f"H₀: μ = {hypothesized_mean}")
print(f"H₁: μ ≠ {hypothesized_mean}")
print(f"Średnia z próbki: {sample_mean:.4f}")
print(f"t = ({sample_mean:.3f} - {hypothesized_mean}) / ({s:.3f}/√{n}) ={t_stat_1samp:.4f}")
print(f"p-value (aproksymacja normalna, n={n}): {p_value_1samp:.6f}")
alpha = 0.05
if p_value_1samp < alpha:
  print(f"WNIOSEK: Odrzucamy H₀ (p < {alpha})")
  print(f"Średnia długość płatka RÓŻNI SIĘ istotnie od {hypothesized_mean} cm")
else:
  print(f"WNIOSEK: Nie odrzucamy H₀ (p >= {alpha})")
# --- PRZYPADEK 2: Test t dwóch prób niezależnych ---
# Porównanie długości płatka: gatunek A (setosa-like) vs B (versicolor-like)
group_A = petal_length[species == "A"]
group_B = petal_length[species == "B"]
# Ręczny test równości wariancji (prosta wersja F-testu: ratio wariancji)
# H₀: σ₁² = σ₂². F = s₁²/s₂² (większa w liczniku)
var_A = group_A.var(ddof=1)
var_B = group_B.var(ddof=1)
F_ratio = max(var_A, var_B) / min(var_A, var_B)
# Dla małych próbek można traktować F>2-3 jako sygnał nierówności wariancji
equal_var = F_ratio < 2.0
print("\n" + "=" * 60)
print("TEST T DWÓCH PRÓB NIEZALEŻNYCH")
print("=" * 60)
print(f"Grupa A: n={len(group_A)}, mean={group_A.mean():.4f}, std={group_A.std(ddof=1):.4f}")
print(f"Grupa B: n={len(group_B)}, mean={group_B.mean():.4f}, std={group_B.std(ddof=1):.4f}")
print(f"Ratio wariancji F = {F_ratio:.3f} → wariancje {'RÓWNE' if equal_var else 'RÓŻNE'}")
# Ręczna t-statystyka
n1, n2 = len(group_A), len(group_B)
m1, m2 = group_A.mean(), group_B.mean()
v1, v2 = group_A.var(ddof=1), group_B.var(ddof=1)
if equal_var:
# Pooled variance
  sp2 = ((n1 - 1) * v1 + (n2 - 1) * v2) / (n1 + n2 - 2)
  se = np.sqrt(sp2 * (1/n1 + 1/n2))
else:
  # Welch's t-test
  se = np.sqrt(v1 / n1 + v2 / n2)
t_stat = (m1 - m2) / se
p_value_normal = two_sided_pvalue_normal(t_stat)
print(f"\nt-statystyka: {t_stat:.4f}")
print(f"p-value (aproksymacja normalna): {p_value_normal:.6e}")
# --- Test permutacyjny jako pełna weryfikacja p-value ---
# Pomieszaj etykiety i sprawdź rozkład null różnic średnich
def permutation_pvalue(a, b, n_permutations=10000, seed=0):
  rng = np.random.default_rng(seed)
  all_data = np.concatenate([a, b])
  observed_diff = a.mean() - b.mean()
  n_a = len(a)
  count_extreme = 0
  for _ in range(n_permutations):
    rng.shuffle(all_data)
    diff = all_data[:n_a].mean() - all_data[n_a:].mean()
    if abs(diff) >= abs(observed_diff):
      count_extreme += 1
  return count_extreme / n_permutations
# Dla pedagogiki: policzmy tylko 2000 permutacji (szybciej)
p_value_perm = permutation_pvalue(group_A, group_B, n_permutations=2000, seed=42)
print(f"p-value (permutation test, 2000 permutacji): {p_value_perm:.6f}")
# --- Cohen's d: wielkość efektu ---
# d = (m₁ - m₂) / sp, gdzie sp = pooled std
sp = np.sqrt(((n1 - 1) * v1 + (n2 - 1) * v2) / (n1 + n2 - 2))
cohen_d = (m1 - m2) / sp
effect = "mały" if abs(cohen_d) < 0.2 else ("średni" if abs(cohen_d) < 0.5 else "duży")
print(f"\nCohen's d = {cohen_d:.4f} (wielkość efektu: {effect})")
# --- ASCII visualizacja: boxplot + histogramy ---
def ascii_boxplot_two(a, b, label_a, label_b, width=50):
  all_data = np.concatenate([a, b])
  d_min, d_max = all_data.min(), all_data.max()
  d_range = max(d_max - d_min, 1e-9)
  def box_row(data, label):
      q1, med, q3 = np.percentile(data, [25, 50, 75])
      lo, hi = data.min(), data.max()
      row = [" "] * width
      # whiskers
      i_lo = int((lo - d_min) / d_range * (width - 1))
      i_hi = int((hi - d_min) / d_range * (width - 1))
      i_q1 = int((q1 - d_min) / d_range * (width - 1))
      i_q3 = int((q3 - d_min) / d_range * (width - 1))
      i_med = int((med - d_min) / d_range * (width - 1))
      for i in range(i_lo, i_hi + 1):
        row[i] = "-"
      for i in range(i_q1, i_q3 + 1):
        row[i] = "="
      row[i_q1] = "|"
      row[i_q3] = "|"
      row[i_med] = "*"
      print(f" {label:<12} |{''.join(row)}| mean={data.mean():.3f}")
  print(f"\nBoxplot (| = kwartyl, * = mediana, - = whisker):")
  box_row(a, label_a)
  box_row(b, label_b)
  print(f" {'':<12} [{d_min:.2f}{' ' * (width - 12)}{d_max:.2f}]")
ascii_boxplot_two(group_A, group_B, "A", "B")
print(f"p-value (permutation) = {p_value_perm:.6f}")
# Ocena istotności
if p_value_perm < 0.001:
  sig = "*** (bardzo silna)"
elif p_value_perm < 0.01:
  sig = "** (silna)"
elif p_value_perm < 0.05:
  sig = "* (istotna)"
else:
  sig = "ns (nieistotna)"
print(f"Istotność: {sig}")

TEST T JEDNEJ PRÓBKI
H₀: μ = 4.0
H₁: μ ≠ 4.0
Średnia z próbki: 3.7969
t = (3.797 - 4.0) / (1.733/√150) =-1.4352
p-value (aproksymacja normalna, n=150): 0.151236
WNIOSEK: Nie odrzucamy H₀ (p >= 0.05)

TEST T DWÓCH PRÓB NIEZALEŻNYCH
Grupa A: n=50, mean=1.5392, std=0.4406
Grupa B: n=50, mean=4.3266, std=0.4630
Ratio wariancji F = 1.104 → wariancje RÓWNE

t-statystyka: -30.8362
p-value (aproksymacja normalna): 0.000000e+00
p-value (permutation test, 2000 permutacji): 0.000000

Cohen's d = -6.1672 (wielkość efektu: duży)

Boxplot (| = kwartyl, * = mediana, - = whisker):
 A            |-------|==*=|-------                              | mean=1.539
 B            |                           -------|===*==|--------| mean=4.327
              [0.54                                      5.43]
p-value (permutation) = 0.000000
Istotność: *** (bardzo silna)


In [ ]:
import pandas as pd
import numpy as np
# Wczytanie datasetu Titanic (pd.read_csv z URL wprowadzony w L11)
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
titanic = pd.read_csv(url)
print("Dataset Titanic:")
print(titanic.head())
# Pytanie: Czy przeżycie zależy od płci?
# H₀: Przeżycie i płeć są niezależne
# H₁: Przeżycie i płeć są zależne
# --- Tabela kontyngencji ---
contingency_table = pd.crosstab(titanic["Sex"], titanic["Survived"], margins=True)
print("\nTabela kontyngencji Sex × Survived:")
print(contingency_table)
contingency_pct = pd.crosstab(titanic["Sex"], titanic["Survived"], normalize="index") * 100
print("\nProcent przeżycia według płci:")
print(contingency_pct.round(2))
# --- Ręczna statystyka chi-kwadrat ---
# Chi² = Σ (O - E)² / E, gdzie E_ij = (row_total_i * col_total_j) / n
observed = pd.crosstab(titanic["Sex"], titanic["Survived"]).values
row_totals = observed.sum(axis=1, keepdims=True)
col_totals = observed.sum(axis=0, keepdims=True)
n_total = observed.sum()
expected = row_totals @ col_totals / n_total
chi2_stat = ((observed - expected) ** 2 / expected).sum()
dof = (observed.shape[0] - 1) * (observed.shape[1] - 1)
print("\nWartości obserwowane O:")
print(observed)
print("\nWartości oczekiwane E (pod H₀ niezależności):")
print(expected.round(2))
print(f"\nChi² = Σ(O-E)²/E = {chi2_stat:.4f}")
print(f"Stopnie swobody: df = (r-1)(c-1) = {dof}")
# --- p-value przez symulację Monte Carlo ---
# Generujemy N tabel pod H₀ (niezależność, stałe totale) i liczymy chi² dla każdej.
# p-value = frakcja symulowanych chi² >= obserwowane chi².
def chi2_monte_carlo_pvalue(observed, n_sim=5000, seed=42):
  rng = np.random.default_rng(seed)
  row_totals = observed.sum(axis=1)
  col_totals = observed.sum(axis=0)
  n = observed.sum()
  expected = np.outer(row_totals, col_totals) / n
  def chi2(table, exp):
    return ((table - exp) ** 2 / exp).sum()
  observed_chi2 = chi2(observed, expected)
  count_extreme = 0
  for _ in range(n_sim):
    # Generujemy tabelę pod H0 przez multinomial z pooled probabilities
    probs = expected.flatten() / n
    table_sim = rng.multinomial(n, probs).reshape(observed.shape)
    exp_sim = np.outer(table_sim.sum(axis=1), table_sim.sum(axis=0)) / n
    exp_sim = np.maximum(exp_sim, 1e-9) # zabezpieczenie przed dzieleniem przez 0
    if chi2(table_sim, exp_sim) >= observed_chi2:
      count_extreme += 1
  return count_extreme / n_sim
p_value = chi2_monte_carlo_pvalue(observed, n_sim=3000, seed=42)
print("\n" + "=" * 60)
print("TEST CHI-KWADRAT (Monte Carlo, 3000 symulacji)")
print("=" * 60)
print(f"H₀: Przeżycie i płeć są niezależne")
print(f"H₁: Przeżycie i płeć są zależne")
print(f"Chi² obserwowane = {chi2_stat:.4f}, df = {dof}")

print(f"p-value (MC) ≈ {p_value:.6f}")
alpha = 0.05
if p_value < alpha:
  print(f"WNIOSEK: Odrzucamy H₀ (p < {alpha})")
  print("Przeżycie ZALEŻY od płci")
else:
  print(f"WNIOSEK: Nie odrzucamy H₀ (p >= {alpha})")
# --- Cramer's V - siła związku ---
min_dim = min(observed.shape[0] - 1, observed.shape[1] - 1)
cramers_v = np.sqrt(chi2_stat / (n_total * min_dim))
strength = "słaby" if cramers_v < 0.1 else ("średni" if cramers_v < 0.3 else "silny")
print(f"\nCramer's V = {cramers_v:.4f} → związek {strength}")
# --- ASCII wizualizacja: stacked bar chart ---
sex_labels = list(contingency_pct.index)
survived_labels = ["Nie przeżył (0)", "Przeżył (1)"]
def ascii_stacked_bars(pct_table, row_labels, col_labels, width=50):
  print("\nProcent przeżycia według płci:")
  print(f" {'':<12} {'0%':<5}{'25%':<13}{'50%':<13}{'75%':<13}{'100%':<5}")
  print(f" {'':<12} +-----+------------+------------+------------+")
  for i, rlabel in enumerate(row_labels):
    row = [" "] * width
    cum = 0
    for j, clabel in enumerate(col_labels):
      pct = pct_table.iloc[i, j]
      n_chars = int(round(pct / 100 * width))
      char = "░" if j == 0 else "█"
      for k in range(cum, min(cum + n_chars, width)):
        row[k] = char
      cum += n_chars
    print(f" {rlabel:<12} |{''.join(row)}| vals:{list(pct_table.iloc[i].round(1).values)}")
  print(f" Legenda: ░ = nie przeżył, █ = przeżył")
ascii_stacked_bars(contingency_pct, sex_labels, survived_labels)
# --- ASCII tabela kontyngencji ---
print("\nTabela kontyngencji (wartości obserwowane):")
print(f" {'':<12} {'Nie przeżył':>12} {'Przeżył':>10}")
for i, label in enumerate(sex_labels):
  print(f" {label:<12} {observed[i,0]:>12} {observed[i,1]:>10}")
# --- DODATKOWY TEST: klasa podróży ---
observed_pclass = pd.crosstab(titanic["Pclass"], titanic["Survived"]).values
row_t = observed_pclass.sum(axis=1, keepdims=True)
col_t = observed_pclass.sum(axis=0, keepdims=True)
n_p = observed_pclass.sum()
expected_p = row_t @ col_t / n_p
chi2_pclass = ((observed_pclass - expected_p) ** 2 / expected_p).sum()
p_pclass = chi2_monte_carlo_pvalue(observed_pclass, n_sim=3000, seed=43)
print("\n" + "=" * 60)
print("TEST CHI-KWADRAT: Pclass × Survived")
print("=" * 60)
print(f"Chi² = {chi2_pclass:.4f}, p-value (MC) ≈ {p_pclass:.6f}")
if p_pclass < alpha:
  print("WNIOSEK: Przeżycie ZALEŻY od klasy podróży")
else:
  print("WNIOSEK: Brak zależności")
pct_by_class = pd.crosstab(titanic["Pclass"], titanic["Survived"], normalize="index") * 100
print("\nProcent przeżycia według klasy:")
print(pct_by_class.round(2))

Dataset Titanic:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   

In [ ]:
import numpy as np
import math
_erf_vec = np.vectorize(math.erf)
# Symulacja A/B testu na stronie internetowej
# Wersja A (kontrolna): stara wersja strony
# Wersja B (testowa): nowa wersja strony
# Metryka: współczynnik konwersji (czy użytkownik dokonał zakupu)
rng = np.random.default_rng(seed=42)
# Parametry eksperymentu
n_visitors_A = 1000
n_visitors_B = 1000
conversion_rate_A = 0.10 # 10% konwersji w A
conversion_rate_B = 0.12 # 12% konwersji w B (improvement!)
# Symulacja danych przez rng.binomial
conversions_A = rng.binomial(1, conversion_rate_A, n_visitors_A)
conversions_B = rng.binomial(1, conversion_rate_B, n_visitors_B)
conv_A, conv_B = int(conversions_A.sum()), int(conversions_B.sum())
rate_A = conv_A / n_visitors_A
rate_B = conv_B / n_visitors_B
print("=" * 60)
print("A/B TEST - ANALIZA KONWERSJI")
print("=" * 60)
print(f"Wersja A (kontrolna): n={n_visitors_A}, konwersje={conv_A}, rate={rate_A*100:.2f}%")
print(f"Wersja B (testowa): n={n_visitors_B}, konwersje={conv_B}, rate={rate_B*100:.2f}%")
print(f"Różnica: {(rate_B - rate_A)*100:+.2f} pp")
print(f"Relative lift: {((rate_B / rate_A - 1) * 100):+.2f}%")
# --- Z-test dla proporcji (ręcznie) ---
# Pooled proportion: p_pool = (c_A + c_B) / (n_A + n_B)
# SE pooled: √(p(1-p) * (1/n_A + 1/n_B))
# z = (p_B - p_A) / SE
p_pool = (conv_A + conv_B) / (n_visitors_A + n_visitors_B)
se_pool = np.sqrt(p_pool * (1 - p_pool) * (1/n_visitors_A + 1/n_visitors_B))
z_stat = (rate_B - rate_A) / se_pool
# p-value two-tailed przez ręczną CDF normalną
def normal_cdf_scalar(z):
  return 0.5 * (1 + float(_erf_vec(z / np.sqrt(2))))
p_value_two_tailed = 2 * (1 - normal_cdf_scalar(abs(z_stat)))
print("\n" + "=" * 60)
print("TEST PROPORCJI (Z-TEST)")
print("=" * 60)
print(f"Pooled proportion: {p_pool:.4f}")
print(f"Standard error (pooled): {se_pool:.6f}")

print(f"Z-statistic: {z_stat:.4f}")
print(f"p-value (two-tailed): {p_value_two_tailed:.6f}")
alpha = 0.05
if p_value_two_tailed < alpha:
  print(f"WNIOSEK: Odrzucamy H₀ (p < {alpha})")
  print("Wersja B ma ISTOTNIE inny współczynnik konwersji niż A")
  if rate_B > rate_A:
    print("Wersja B jest LEPSZA - warto wdrożyć!")
else:
  print(f"WNIOSEK: Nie odrzucamy H₀ (p >= {alpha})")
  print("Brak istotnych różnic - potrzeba więcej danych")
# --- 95% przedział ufności dla różnicy proporcji ---
# SE_diff (unpooled, Wald CI): √(p_A(1-p_A)/n_A + p_B(1-p_B)/n_B)
# CI: (p_B - p_A) ± 1.96 * SE_diff
se_diff = np.sqrt(rate_A * (1 - rate_A) / n_visitors_A +
rate_B * (1 - rate_B) / n_visitors_B)
ci_lower = (rate_B - rate_A) - 1.96 * se_diff
ci_upper = (rate_B - rate_A) + 1.96 * se_diff
print(f"\n95% przedział ufności dla różnicy (B - A):")
print(f" [{ci_lower*100:+.2f}%, {ci_upper*100:+.2f}%] pp")
# --- ASCII bar chart konwersji ---
print("\n" + "=" * 60)
print("Wizualizacja ASCII - słupki konwersji")
print("=" * 60)
max_rate = max(rate_A, rate_B) * 100
width = 50
bar_A = int(rate_A * 100 / max_rate * width)
bar_B = int(rate_B * 100 / max_rate * width)
print(f" A (kontrolna) | {'█' * bar_A}{' ' * (width - bar_A)} | {rate_A*100:.2f}%")
print(f" B (testowa) | {'█' * bar_B}{' ' * (width - bar_B)} | {rate_B*100:.2f}%")
# Istotność
if p_value_two_tailed < 0.001:
  sig = "***"
elif p_value_two_tailed < 0.01:
  sig = "**"
elif p_value_two_tailed < 0.05:
  sig = "*"
else:
  sig = "ns"
print(f"\n Istotność: {sig} (p = {p_value_two_tailed:.4f})")
# --- ASCII wizualizacja CI ---
print("\nPrzedział ufności 95% dla różnicy (B-A):")
ci_width = 50
# Rozpiętość wokół zera od -max do +max
max_abs = max(abs(ci_lower), abs(ci_upper), 0.01) * 1.2 * 100
ci_low_pct, ci_hi_pct = ci_lower * 100, ci_upper * 100
zero_pos = int(max_abs / (2 * max_abs) * ci_width)
lo_pos = int((ci_low_pct + max_abs) / (2 * max_abs) * ci_width)
hi_pos = int((ci_hi_pct + max_abs) / (2 * max_abs) * ci_width)
diff_pos = int(((rate_B - rate_A) * 100 + max_abs) / (2 * max_abs) * ci_width)
row = [" "] * (ci_width + 1)
for i in range(lo_pos, hi_pos + 1):
  row[i] = "═"
row[zero_pos] = "│" # H0: brak różnicy
row[diff_pos] = "●"
row[lo_pos] = "├"
row[hi_pos] = "┤"
print(f" [{ci_low_pct:+.2f}pp] {''.join(row)} [{ci_hi_pct:+.2f}pp]")
print(f" │ = brak różnicy (H₀), ● = obserwowana różnica, ═ = 95% CI")

A/B TEST - ANALIZA KONWERSJI
Wersja A (kontrolna): n=1000, konwersje=98, rate=9.80%
Wersja B (testowa): n=1000, konwersje=128, rate=12.80%
Różnica: +3.00 pp
Relative lift: +30.61%

TEST PROPORCJI (Z-TEST)
Pooled proportion: 0.1130
Standard error (pooled): 0.014158
Z-statistic: 2.1189
p-value (two-tailed): 0.034101
WNIOSEK: Odrzucamy H₀ (p < 0.05)
Wersja B ma ISTOTNIE inny współczynnik konwersji niż A
Wersja B jest LEPSZA - warto wdrożyć!

95% przedział ufności dla różnicy (B - A):
 [+0.23%, +5.77%] pp

Wizualizacja ASCII - słupki konwersji
 A (kontrolna) | ██████████████████████████████████████             | 9.80%
 B (testowa) | ██████████████████████████████████████████████████ | 12.80%

 Istotność: * (p = 0.0341)

Przedział ufności 95% dla różnicy (B-A):
 [+0.23pp]                          ├═════════●═════════┤      [+5.77pp]
 │ = brak różnicy (H₀), ● = obserwowana różnica, ═ = 95% CI


In [ ]:
import pandas as pd
import numpy as np
import math
_erf_vec = np.vectorize(math.erf)
def make_synthetic_wine(n_samples=1600, seed=42):
  """
  Kanoniczny syntetyczny dataset wina (wzorowany na UCI Wine Quality - red).
  11 cech fizykochemicznych + quality (3..8).
  """
  rng = np.random.default_rng(seed)
  alcohol = rng.normal(10.4, 1.1, size=n_samples).clip(8.4, 14.9)
  volatile_acidity = rng.gamma(shape=2.0, scale=0.25, size=n_samples).clip(0.1, 1.6)
  sulphates = rng.gamma(shape=5.0, scale=0.13, size=n_samples).clip(0.33, 2.0)
  citric_acid = rng.uniform(0.0, 0.75, size=n_samples)
  residual_sugar = rng.gamma(shape=2.0, scale=1.2, size=n_samples).clip(0.9, 15.0)
  chlorides = rng.gamma(shape=3.0, scale=0.03, size=n_samples).clip(0.012, 0.61)
  free_sulfur_dioxide = rng.gamma(shape=3.0, scale=5.0, size=n_samples).clip(1, 72)
  total_sulfur_dioxide = free_sulfur_dioxide * rng.uniform(2.0, 4.5, size=n_samples)
  density = rng.normal(0.9967, 0.0019, size=n_samples)
  pH = rng.normal(3.31, 0.154, size=n_samples).clip(2.74, 4.01)
  fixed_acidity = rng.gamma(shape=7.0, scale=1.2, size=n_samples).clip(4.6, 15.9)
  # quality: liniowa kombinacja + szum + zaokrąglenie do skali 3..8
  quality_raw = (
  5.0
  + 0.32 * (alcohol - 10.4)
  - 1.15 * (volatile_acidity - 0.5)
  + 0.57 * (sulphates - 0.65)
  + 0.05 * (citric_acid - 0.3)
  + rng.normal(0, 0.5, size=n_samples)
  )
  quality = np.round(np.clip(quality_raw, 3, 8)).astype(int)
  return pd.DataFrame({
      "fixed acidity": fixed_acidity, "volatile acidity": volatile_acidity,
  "citric acid": citric_acid, "residual sugar": residual_sugar,
  "chlorides": chlorides, "free sulfur dioxide": free_sulfur_dioxide,
  "total sulfur dioxide": total_sulfur_dioxide, "density": density,
  "pH": pH, "sulphates": sulphates, "alcohol": alcohol, "quality": quality,
  })
wine = make_synthetic_wine(seed=42)
print("Syntetyczny dataset Wine Quality:")
print(wine.head())
print(f"\nShape: {wine.shape}")
print(f"\nOpis statystyczny:")
print(wine.describe().round(3))
# --- ANALIZA ROZKŁADÓW (test normalności Jarque-Bera) ---
def skewness(x):
  m = x.mean(); s = x.std(ddof=0)
  return ((x - m) ** 3).mean() / (s ** 3)
def kurtosis_excess(x):
  m = x.mean(); s = x.std(ddof=0)
  return ((x - m) ** 4).mean() / (s ** 4) - 3
print("\n" + "=" * 60)
print("TEST NORMALNOŚCI (Jarque-Bera)")
print("=" * 60)
for col in wine.columns:
  data = wine[col].values.astype(float)
  sk = skewness(data)
  ku = kurtosis_excess(data)
  jb = len(data) / 6.0 * (sk ** 2 + (ku ** 2) / 4)
  p = math.exp(-jb / 2) # p-value dla chi²(2)
  verdict = "(normalny)" if p > 0.05 else "(nienormalny)"
  print(f" {col:<22} JB={jb:>8.2f} p≈{p:.4f} {verdict}")
# --- ASCII histogramy kluczowych cech ---
def ascii_mini_histogram(data, label, bins=24, height=6):
  counts, edges = np.histogram(data, bins=bins)
  max_count = max(counts.max(), 1)
  scaled = np.round(counts / max_count * height).astype(int)
  m, md = data.mean(), np.median(data)
  sk = skewness(data)
  print(f"\n {label:<20} mean={m:.3f} median={md:.3f} skew={sk:+.2f}")
  for h in range(height, 0, -1):
    print(" " + "".join("█" if s >= h else " " for s in scaled))
  print(" " + "─" * bins)
  print(f" [{edges[0]:.2f}" + " " * (bins - 14) + f"{edges[-1]:.2f}]")
key_features = ["alcohol", "pH", "sulphates", "residual sugar", "density", "quality"]
for feature in key_features:
  ascii_mini_histogram(wine[feature].values, feature)
# --- ANALIZA KORELACJI ---
print("\n" + "=" * 60)
print("ANALIZA KORELACJI")
print("=" * 60)
corr_matrix = np.corrcoef(wine.values.T)
cols = list(wine.columns)
# Korelacje z quality
quality_idx = cols.index("quality")
quality_corr = corr_matrix[quality_idx]
order = np.argsort(-np.abs(quality_corr))
print("\nKorelacje z quality (sortowane po |r|):")
for i in order:
  if cols[i] != "quality":
    print(f" {cols[i]:<22} r = {quality_corr[i]:+.4f}")
# Top 3 najwyższe korelacje z quality
top3_idx = [i for i in order if cols[i] != "quality"][:3]
top3_names = [cols[i] for i in top3_idx]
print(f"\nTop 3 cechy skorelowane z quality: {top3_names}")
# ASCII scatter dla każdej top-3 cechy
def ascii_scatter_mini(x, y, title, rows=10, cols=40):
  x_min, x_max = x.min(), x.max()
  y_min, y_max = y.min(), y.max()
  x_range = max(x_max - x_min, 1e-9)
  y_range = max(y_max - y_min, 1e-9)
  grid = [[" "] * cols for _ in range(rows)]
  for xi, yi in zip(x, y):
    c = int((xi - x_min) / x_range * (cols - 1))
    r_ = rows - 1 - int((yi - y_min) / y_range * (rows - 1))
    if grid[r_][c] == " ":
      grid[r_][c] = "·"
    elif grid[r_][c] == "·":
      grid[r_][c] = "*"
    else:
      grid[r_][c] = "#"
  slope, intercept = np.polyfit(x, y, 1)
  for c in range(cols):
    xi = x_min + c / (cols - 1) * x_range
    yi = slope * xi + intercept
    r_ = rows - 1 - int((yi - y_min) / y_range * (rows - 1))
    if 0 <= r_ < rows and grid[r_][c] == " ":
      grid[r_][c] = "─"
  r, _ = np.corrcoef(x, y)[0, 1], None
  print(f"\n {title} r={np.corrcoef(x, y)[0, 1]:+.3f}")
  for row in grid:
    print(" " + "".join(row))
for name in top3_names:
  ascii_scatter_mini(wine[name].values, wine["quality"].values,title=f"{name} vs quality")
# --- TESTOWANIE HIPOTEZ ---
print("\n" + "=" * 60)
print("TESTOWANIE HIPOTEZ")
print("=" * 60)
# Czy wina wysokiej jakości (quality >= 7) mają wyższą zawartość alkoholu?
high_q = wine[wine["quality"] >= 7]["alcohol"].values
low_q = wine[wine["quality"] < 7]["alcohol"].values
print(f"Wina quality >= 7: n={len(high_q)}, mean={high_q.mean():.4f}")
print(f"Wina quality < 7: n={len(low_q)}, mean={low_q.mean():.4f}")
# Ręczny Welch's t-test
n1, n2 = len(high_q), len(low_q)
m1, m2 = high_q.mean(), low_q.mean()
v1, v2 = high_q.var(ddof=1), low_q.var(ddof=1)
se = np.sqrt(v1 / n1 + v2 / n2)
t = (m1 - m2) / se
p_val = 2 * (1 - 0.5 * (1 + float(_erf_vec(abs(t) / np.sqrt(2)))))
print(f"\nt-statystyka = {t:.4f}")
print(f"p-value (aproksymacja normalna) = {p_val:.6e}")
if p_val < 0.05:
  print("WNIOSEK: Wina wysokiej jakości mają ISTOTNIE wyższą zawartość alkoholu")
else:
  print("WNIOSEK: Brak istotnych różnic w zawartości alkoholu")
# ASCII boxplot porównujący dwie grupy
def ascii_boxplot(data_list, labels, width=50):
  all_data = np.concatenate(data_list)
  d_min, d_max = all_data.min(), all_data.max()
  d_range = max(d_max - d_min, 1e-9)
  print(f"\n Boxplot alkohol według jakości (| kwartyle, * mediana):")
  for data, label in zip(data_list, labels):
    q1, med, q3 = np.percentile(data, [25, 50, 75])
    row = [" "] * width
    i_lo = int((data.min() - d_min) / d_range * (width - 1))
    i_hi = int((data.max() - d_min) / d_range * (width - 1))
    i_q1 = int((q1 - d_min) / d_range * (width - 1))
    i_q3 = int((q3 - d_min) / d_range * (width - 1))
    i_med = int((med - d_min) / d_range * (width - 1))
    for i in range(i_lo, i_hi + 1):
      row[i] = "-"
    for i in range(i_q1, i_q3 + 1):
      row[i] = "="
    row[i_q1], row[i_q3], row[i_med] = "|", "|", "*"
    print(f" {label:<14} |{''.join(row)}| mean={data.mean():.2f}")
  print(f" {'':<14} [{d_min:.2f}{' ' * (width - 14)}{d_max:.2f}]")
ascii_boxplot([low_q, high_q], ["Quality < 7", "Quality >= 7"])
if p_val < 0.001:
  sig = "***"
elif p_val < 0.01:
  sig = "**"
elif p_val < 0.05:
  sig = "*"
else:
  sig = "ns"
print(f"\n Istotność: {sig} (p = {p_val:.4e})")

Syntetyczny dataset Wine Quality:
   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0       5.337116          0.399187     0.152028        3.233766   0.273758   
1       5.555572          0.972553     0.016483        5.464401   0.024270   
2       7.879531          0.323766     0.735677        4.863200   0.176117   
3       5.640307          0.603101     0.194360        0.900000   0.088048   
4       6.335022          0.350284     0.444831        0.900000   0.059808   

   free sulfur dioxide  total sulfur dioxide   density        pH  sulphates  \
0            19.315469             66.358296  0.994562  3.086856   0.601784   
1            17.541303             74.924045  0.997984  3.148137   0.657098   
2             3.815322             12.458480  0.999426  3.362340   1.303008   
3             2.198062              8.108279  0.999149  3.420159   0.784769   
4             7.691284             28.947739  0.998058  3.186875   0.483775   

     alcohol  quality 

In [ ]:
#zadanie 1
import numpy as np
import math
# Parametry rozkładu normalnego
mean = 50 # średnia (μ)
std = 10 # odchylenie standardowe (σ)
# Generowanie danych z rozkładu normalnego
rng = np.random.default_rng(seed=42)
Dane = rng.normal(loc=mean, scale=std, size=1000)
# --- Ręczne implementacje PDF i CDF ---
def normal_pdf(x, mu, sigma):
  """Gęstość rozkładu normalnego: f(x) = 1/(σ√(2π)) * exp(-0.5 * ((x-μ)/σ)²)."""
  return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))
_erf_vec = np.vectorize(math.erf)
def normal_cdf(x, mu, sigma):
  """Dystrybuanta rozkładu normalnego przez math.erf."""
  return 0.5 * (1.0 + _erf_vec((x - mu) / (sigma * np.sqrt(2))))
# --- ASCII histogram (bez matplotlib) ---
def ascii_histogram(data, bins=30, height=10, label="Histogram"):
  counts, edges = np.histogram(data, bins=bins)
  max_count = counts.max() if counts.max() > 0 else 1
  scaled = np.round(counts / max_count * height).astype(int)
  print(f"\n{label} (max={max_count}):")
  for h in range(height, 0, -1):
    row = "".join("█" if s >= h else " " for s in scaled)
    print(f" {row}")
    print(" " + "─" * bins)
    print(f" {edges[0]:>5.1f}{' ' * (bins - 12)}{edges[-1]:>5.1f}")
# Empiryczny histogram próbki
print(f"Rozkład próbki w granicach <40,60> stanowi:{((Dane >= 40) & (Dane <= 60)).mean():.2%} całej próby ")
print(f"Rozkład próbki według normal_cdf w granicach <40,60> stanowi:{normal_cdf(60,50,10)-normal_cdf(40,50,10):.2%} całej próby ")
ascii_histogram(Dane, bins=20, height=10, label="N(50,10)")
print(f"|---[40]=[50]=[60]---|")

Rozkład próbki w granicach <40,60> stanowi:68.80% całej próby 
Rozkład próbki według normal_cdf w granicach <40,60> stanowi:68.27% całej próby 

N(50,10) (max=160):
            █        
 ────────────────────
  13.5         81.8
            █        
 ────────────────────
  13.5         81.8
          ███        
 ────────────────────
  13.5         81.8
          ███        
 ────────────────────
  13.5         81.8
         █████       
 ────────────────────
  13.5         81.8
        ███████      
 ────────────────────
  13.5         81.8
        ███████      
 ────────────────────
  13.5         81.8
       █████████     
 ────────────────────
  13.5         81.8
      ███████████    
 ────────────────────
  13.5         81.8
    ██████████████   
 ────────────────────
  13.5         81.8
|---[40]=[50]=[60]---|


In [ ]:
#Zadanie 2 – Obliczanie prawdopodobieństw (ręcznie)
import numpy as np
import math
# Parametry rozkładu normalnego
mean = 100 # średnia (μ)
std = 15 # odchylenie standardowe (σ)
# Generowanie danych z rozkładu normalnego
rng = np.random.default_rng(seed=42)
Dane = rng.normal(loc=mean, scale=std, size=1000)
def normal_cdf(x, mu, sigma):
  """Dystrybuanta rozkładu normalnego przez math.erf."""
  return 0.5 * (1.0 + _erf_vec((x - mu) / (sigma * np.sqrt(2))))
# --- ASCII histogram (bez matplotlib) ---
def ascii_histogram(data, bins=30, height=10, label="Histogram"):
  counts, edges = np.histogram(data, bins=bins)
  max_count = counts.max() if counts.max() > 0 else 1
  scaled = np.round(counts / max_count * height).astype(int)
  print(f"\n{label} (max={max_count}):")
  for h in range(height, 0, -1):
    row = "".join("█" if s >= h else " " for s in scaled)
    print(f" {row}")
    print(" " + "─" * bins)
    print(f" {edges[0]:>5.1f}{' ' * (bins - 12)}{edges[-1]:>5.1f}")
# Empiryczny histogram próbki
print(f"P(X < 85) = {normal_cdf(85, 100, 15):.2%}")
print(f"P(X > 120) = {1 - normal_cdf(120, 100, 15):.2%}")
print(f"P(90 < X < 110) = {normal_cdf(110, 100, 15) - normal_cdf(90, 100, 15):.2%}")
# 15,87% wygenerowanych wartości powinno przyjmowac wartości mniejsze od 85
# 9,12% wygenerowanych wartości powinno przyjmowac wartości większe od 120
# 49,50% wygenerowanych wartości powinno przyjmowac wartości z przedziału od 90 do 110

ascii_histogram(Dane, bins=20, height=10, label="N(100,15)")


P(X < 85) = 15.87%
P(X > 120) = 9.12%
P(90 < X < 110) = 49.50%

N(100,15) (max=160):
            █        
 ────────────────────
  45.3        147.7
            █        
 ────────────────────
  45.3        147.7
          ███        
 ────────────────────
  45.3        147.7
          ███        
 ────────────────────
  45.3        147.7
         █████       
 ────────────────────
  45.3        147.7
        ███████      
 ────────────────────
  45.3        147.7
        ███████      
 ────────────────────
  45.3        147.7
       █████████     
 ────────────────────
  45.3        147.7
      ███████████    
 ────────────────────
  45.3        147.7
    ██████████████   
 ────────────────────
  45.3        147.7


In [ ]:
# Zadanie 3 – Analiza rozkładu kwiatów
import numpy as np
def make_synthetic_flowers(seed=42):
  """Generuje syntetyczny dataset kwiatów (150 × 4) z 3 gatunkami.
  Zwraca:
  X: macierz cech (150, 4) – sepal_length, sepal_width, petal_length, petal_width
  (cm)
  y: wektor etykiet (150,) – 0, 1, 2 (3 gatunki po 50 obserwacji)
  feature_names: nazwy cech
  target_names: nazwy gatunków
  """
  rng = np.random.default_rng(seed)
  n_per_class = 50
  # 3 gatunki o różnych średnich parametrach (cm)
  centers = np.array([
  [5.0, 3.4, 1.5, 0.2], # Gatunek A (małe płatki)
  [5.9, 2.8, 4.3, 1.3], # Gatunek B (średnie płatki)
  [6.6, 3.0, 5.6, 2.0], # Gatunek C (duże płatki)
  ])
  stds = np.array([0.35, 0.35, 0.45, 0.25]) # rozrzut per cecha
  X_list, y_list = [], []
  for class_idx, center in enumerate(centers):
    samples = rng.normal(loc=center, scale=stds, size=(n_per_class, 4))
    X_list.append(samples)
    y_list.append(np.full(n_per_class, class_idx))
  X = np.vstack(X_list)
  y = np.concatenate(y_list)
  feature_names = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
  target_names = np.array(['gatunek_A', 'gatunek_B', 'gatunek_C'])
  return X, y, feature_names, target_names

def jarque_bera_manual(x):
  n = len(x)
  sk = skewness(x)
  ku = kurtosis_excess(x)
  jb = n / 6.0 * (sk ** 2 + (ku ** 2) / 4.0)
  # Chi-square(2) has CDF F(x) = 1 - exp(-x/2). p-value = 1 - F(jb) = exp(-jb/2).
  p_value = math.exp(-jb / 2)
  return jb, p_value, sk, ku


def ascii_histogram(data, bins=30, height=10, label="Histogram"):
  counts, edges = np.histogram(data, bins=bins)
  max_count = counts.max() if counts.max() > 0 else 1
  scaled = np.round(counts / max_count * height).astype(int)
  print(f"\n{label} (max={max_count}):")
  for h in range(height, 0, -1):
    row = "".join("█" if s >= h else " " for s in scaled)
    print(f" {row}")
    print(" " + "─" * bins)
    print(f" {edges[0]:>5.1f}{' ' * (bins - 12)}{edges[-1]:>5.1f}")

def normal_ppf(p, iterations=60):
  """Odwrotna CDF N(0,1) przez bisekcję (wystarczająco dokładna do wizualizacji)."""
  p = np.asarray(p)
  lo = np.full_like(p, -8.0, dtype=float)
  hi = np.full_like(p, 8.0, dtype=float)
  for _ in range(iterations):
    mid = (lo + hi) / 2
    cdf = normal_cdf(mid,0,1)
    lo = np.where(cdf < p, mid, lo)
    hi = np.where(cdf >= p, mid, hi)
  return (lo + hi) / 2

def ascii_qq_plot(data, name, rows=12, cols=40):
  """ASCII Q-Q plot: kwantyle empiryczne vs teoretyczne N(0,1)."""
  n = len(data)
  sorted_data = np.sort(data)
  # Standaryzacja: z = (x-mean)/std
  z_empirical = (sorted_data - sorted_data.mean()) / sorted_data.std(ddof=0)
  # Teoretyczne kwantyle: p_i = (i - 0.5)/n
  p = (np.arange(1, n + 1) - 0.5) / n
  z_theoretical = normal_ppf(p)
  # Rysujemy siatkę rows×cols
  x_min, x_max = z_theoretical.min(), z_theoretical.max()
  y_min, y_max = z_empirical.min(), z_empirical.max()
  y_range = max(y_max - y_min, 1e-9)
  x_range = max(x_max - x_min, 1e-9)
  grid = [[" "] * cols for _ in range(rows)]
  for xt, ye in zip(z_theoretical, z_empirical):
    col = int((xt - x_min) / x_range * (cols - 1))
    row = rows - 1 - int((ye - y_min) / y_range * (rows - 1))
    grid[row][col] = "*"
    # Linia idealna y=x: rysujemy kropkami
  for c in range(cols):
    xt = x_min + c / (cols - 1) * x_range
    row = rows - 1 - int((xt - y_min) / y_range * (rows - 1))
    if 0 <= row < rows and grid[row][c] == " ":
      grid[row][c] = "·"
  print(f"\nQ-Q plot: {name} (* = dane, · = idealna linia y=x)")
  for row in grid:
    print(" |" + "".join(row) + "|")
  print(" " + "─" * (cols + 2))
  print(f" kwantyle teoretyczne N(0,1): {x_min:+.2f} .. {x_max:+.2f}")


# Wczytanie datasetu
X, y, feature_names, target_names = make_synthetic_flowers()
print("Kształt macierzy cech:", X.shape) # (150, 4): 150 obserwacji, 4 cechy
print("Kształt wektora etykiet:", y.shape) # (150,): 150 etykiet
print("Nazwy cech:", feature_names)
print("Gatunki:", target_names)
# Analiza wszystkich gatunków
sepal_length = X[:, 0]

groups = [("Wszystkie gatunki", sepal_length)]

# Analiza pojedynczych gatunków
for i, name in enumerate(target_names):
    species_data = X[y == i, 0]
    groups.append((name, species_data))

# Test Jarque-Bera, histogram i Q-Q plot
for name, data in groups:

    jb, p_value, sk, ku = jarque_bera_manual(data)

    print("\n" + "=" * 50)
    print(f"ANALIZA: {name}")
    print("=" * 50)

    print(f"Liczba obserwacji: {len(data)}")
    print(f"Średnia: {data.mean():.4f}")
    print(f"Statystyka JB: {jb:.4f}")
    print(f"p-value: {p_value:.4f}")
    print(f"Skośność: {sk:.4f}")
    print(f"Kurtoza excess: {ku:.4f}")

    if p_value < 0.05:
        print("Wniosek: odrzucamy H0 – rozkład odbiega od normalnego.")
    else:
        print("Wniosek: brak podstaw do odrzucenia H0.")

    ascii_histogram(data, label=name)
    ascii_qq_plot(data, name)

    #Dla wszystkich kategorii rozkład nie jest normalny ale dla każdego pojedynczego gatunku już tak

Kształt macierzy cech: (150, 4)
Kształt wektora etykiet: (150,)
Nazwy cech: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
Gatunki: ['gatunek_A' 'gatunek_B' 'gatunek_C']

ANALIZA: Wszystkie gatunki
Liczba obserwacji: 150
Średnia: 5.8201
Statystyka JB: 7.5351
p-value: 0.0231
Skośność: -0.0666
Kurtoza excess: -1.0899
Wniosek: odrzucamy H0 – rozkład odbiega od normalnego.

Wszystkie gatunki (max=12):
                       █       
 ──────────────────────────────
   4.3                    7.3
                       █       
 ──────────────────────────────
   4.3                    7.3
         ██      █     █       
 ──────────────────────────────
   4.3                    7.3
      █  ██      ██   ██       
 ──────────────────────────────
   4.3                    7.3
      █  ██     ███   ██       
 ──────────────────────────────
   4.3                    7.3
      █ ███  █  ███   ██       
 ──────────────────────────────
   4.3                    7.3
      █ ███  █  ███

In [10]:
#Zadanie 4 – Korelacja Pearsona

import numpy as np
import math

_erf_vec = np.vectorize(math.erf)

def make_synthetic_flowers(seed=42):
  """Generuje syntetyczny dataset kwiatów (150 × 4) z 3 gatunkami.
  Zwraca:
  X: macierz cech (150, 4) – sepal_length, sepal_width, petal_length, petal_width
  (cm)
  y: wektor etykiet (150,) – 0, 1, 2 (3 gatunki po 50 obserwacji)
  feature_names: nazwy cech
  target_names: nazwy gatunków
  """
  rng = np.random.default_rng(seed)
  n_per_class = 50
  # 3 gatunki o różnych średnich parametrach (cm)
  centers = np.array([
  [5.0, 3.4, 1.5, 0.2], # Gatunek A (małe płatki)
  [5.9, 2.8, 4.3, 1.3], # Gatunek B (średnie płatki)
  [6.6, 3.0, 5.6, 2.0], # Gatunek C (duże płatki)
  ])
  stds = np.array([0.35, 0.35, 0.45, 0.25]) # rozrzut per cecha
  X_list, y_list = [], []
  for class_idx, center in enumerate(centers):
    samples = rng.normal(loc=center, scale=stds, size=(n_per_class, 4))
    X_list.append(samples)
    y_list.append(np.full(n_per_class, class_idx))
  X = np.vstack(X_list)
  y = np.concatenate(y_list)
  feature_names = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
  target_names = np.array(['gatunek_A', 'gatunek_B', 'gatunek_C'])
  return X, y, feature_names, target_names

def jarque_bera_manual(x):
  n = len(x)
  sk = skewness(x)
  ku = kurtosis_excess(x)
  jb = n / 6.0 * (sk ** 2 + (ku ** 2) / 4.0)
  # Chi-square(2) has CDF F(x) = 1 - exp(-x/2). p-value = 1 - F(jb) = exp(-jb/2).
  p_value = math.exp(-jb / 2)
  return jb, p_value, sk, ku


def ascii_histogram(data, bins=30, height=10, label="Histogram"):
  counts, edges = np.histogram(data, bins=bins)
  max_count = counts.max() if counts.max() > 0 else 1
  scaled = np.round(counts / max_count * height).astype(int)
  print(f"\n{label} (max={max_count}):")
  for h in range(height, 0, -1):
    row = "".join("█" if s >= h else " " for s in scaled)
    print(f" {row}")
    print(" " + "─" * bins)
    print(f" {edges[0]:>5.1f}{' ' * (bins - 12)}{edges[-1]:>5.1f}")

def normal_ppf(p, iterations=60):
  """Odwrotna CDF N(0,1) przez bisekcję (wystarczająco dokładna do wizualizacji)."""
  p = np.asarray(p)
  lo = np.full_like(p, -8.0, dtype=float)
  hi = np.full_like(p, 8.0, dtype=float)
  for _ in range(iterations):
    mid = (lo + hi) / 2
    cdf = normal_cdf(mid,0,1)
    lo = np.where(cdf < p, mid, lo)
    hi = np.where(cdf >= p, mid, hi)
  return (lo + hi) / 2

def ascii_qq_plot(data, name, rows=12, cols=40):
  """ASCII Q-Q plot: kwantyle empiryczne vs teoretyczne N(0,1)."""
  n = len(data)
  sorted_data = np.sort(data)
  # Standaryzacja: z = (x-mean)/std
  z_empirical = (sorted_data - sorted_data.mean()) / sorted_data.std(ddof=0)
  # Teoretyczne kwantyle: p_i = (i - 0.5)/n
  p = (np.arange(1, n + 1) - 0.5) / n
  z_theoretical = normal_ppf(p)
  # Rysujemy siatkę rows×cols
  x_min, x_max = z_theoretical.min(), z_theoretical.max()
  y_min, y_max = z_empirical.min(), z_empirical.max()
  y_range = max(y_max - y_min, 1e-9)
  x_range = max(x_max - x_min, 1e-9)
  grid = [[" "] * cols for _ in range(rows)]
  for xt, ye in zip(z_theoretical, z_empirical):
    col = int((xt - x_min) / x_range * (cols - 1))
    row = rows - 1 - int((ye - y_min) / y_range * (rows - 1))
    grid[row][col] = "*"
    # Linia idealna y=x: rysujemy kropkami
  for c in range(cols):
    xt = x_min + c / (cols - 1) * x_range
    row = rows - 1 - int((xt - y_min) / y_range * (rows - 1))
    if 0 <= row < rows and grid[row][c] == " ":
      grid[row][c] = "·"
  print(f"\nQ-Q plot: {name} (* = dane, · = idealna linia y=x)")
  for row in grid:
    print(" |" + "".join(row) + "|")
  print(" " + "─" * (cols + 2))
  print(f" kwantyle teoretyczne N(0,1): {x_min:+.2f} .. {x_max:+.2f}")

def pearson_corr(x, y):
  """Ręczny Pearson: r = cov(x,y) / (std(x) * std(y))."""
  x = np.asarray(x, dtype=float)
  y = np.asarray(y, dtype=float)
  xm, ym = x.mean(), y.mean()
  cov = ((x - xm) * (y - ym)).mean()
  return cov / (x.std(ddof=0) * y.std(ddof=0))

def ascii_scatter(x, y, title, rows=14, cols=50):
  x_min, x_max = x.min(), x.max()
  y_min, y_max = y.min(), y.max()
  x_range = max(x_max - x_min, 1e-9)
  y_range = max(y_max - y_min, 1e-9)
  grid = [[" "] * cols for _ in range(rows)]
  for xi, yi in zip(x, y):
    c = int((xi - x_min) / x_range * (cols - 1))
    r_ = rows - 1 - int((yi - y_min) / y_range * (rows - 1))
    if grid[r_][c] == " ":
      grid[r_][c] = "·"
    elif grid[r_][c] == "·":
      grid[r_][c] = "*"
    else:
      grid[r_][c] = "#" # gęsty punkt
  # Linia regresji
  slope, intercept = np.polyfit(x, y, 1)
  for c in range(cols):
    xi = x_min + c / (cols - 1) * x_range
    yi = slope * xi + intercept
    r_ = rows - 1 - int((yi - y_min) / y_range * (rows - 1))
    if 0 <= r_ < rows and grid[r_][c] in (" ", "·"):
      grid[r_][c] = "─"
  print(f"\n{title}")
  print(f"y = {slope:.3f}·x + {intercept:.3f}")
  for row in grid:
    print(" |" + "".join(row) + "|")
  print(" " + "─" * (cols + 2))
  print(f" x: [{x_min:.2f}, {x_max:.2f}] y: [{y_min:.2f}, {y_max:.2f}]")

# Wczytanie datasetu
X, y, feature_names, target_names = make_synthetic_flowers()
print("Kształt macierzy cech:", X.shape) # (150, 4): 150 obserwacji, 4 cechy
print("Kształt wektora etykiet:", y.shape) # (150,): 150 etykiet
print("Nazwy cech:", feature_names)
print("Gatunki:", target_names)
# Analiza wszystkich gatunków
petal_length = X[:, 2]
petal_width = X[:,3]

Korelacja= pearson_corr(petal_length,petal_width)
print(Korelacja) # Jest ona ujemna więc wzrast z wzrostem X, Y maleje

def corr_pvalue_approx(r, n):
  """Aproksymacja normalna p-value dla Pearson r lub Spearman ρ."""
  t = r * np.sqrt((n - 2) / max(1 - r ** 2, 1e-12))
  return 2 * (1 - 0.5 * (1 + float(_erf_vec(abs(t) / np.sqrt(2)))))

pvalue=corr_pvalue_approx(Korelacja,150)
if(pvalue<0.05):
  print("Hipoteza 0 jest odrzucona, zmienne są zależne pvalue=",pvalue)
else:
  print("Brak podstaw do obalenia hipotezy 0")
ascii_scatter(petal_length, petal_width, title="petal_length vs petal_width", rows=20, cols=60)

if(abs(Korelacja)>0.7):
  print("Korelacja silna")
elif(abs(Korelacja)>0.4):
  print("Korelacja średnia")
else:
  print("Korelacja słaba")

Kształt macierzy cech: (150, 4)
Kształt wektora etykiet: (150,)
Nazwy cech: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
Gatunki: ['gatunek_A' 'gatunek_B' 'gatunek_C']
0.9375942871198966
Hipoteza 0 jest odrzucona, zmienne są zależne pvalue= 0.0

petal_length vs petal_width
y = 0.418·x + -0.426
 |                                                ·           |
 |                                                            |
 |                                               ·    *      ─|
 |                                               ·* ·     ─── |
 |                                         · ··* *· ··────   ·|
 |                                         ··    ·**── ·*     |
 |                                      *    #  *── · *       |
 |                               ·  · ·  ·  ───* · *          |
 |                                  ·· · ──*       ·          |
 |                            ·   ··*─#──·*##                 |
 |                          ·  ·  ───  · *   · 

In [19]:
#Zadanie 5 – Macierz korelacji

import pandas as pd
import numpy as np

def ascii_heatmap(M, labels):
  n = len(labels)
  # Skróć etykiety do 7 znaków
  short = [l[:7] for l in labels]
  print("\nASCII heatmap macierzy korelacji (legenda: █≥.75 ▓≥.5 ▒≥.25 ░≥.05 ·≈0 -/=≤0#/@≤-.5):")
  header = " " * 9 + "".join(f"{s:>8}" for s in short)
  print(header)
  for i in range(n):
    row = f"{short[i]:>10} "
    for j in range(n):
      row += f" {corr_to_char(M[i,j])} {M[i,j]:+4.2f}"
    print(row)

def corr_to_char(r):
  """Mapuje r ∈ [-1, 1] na znak: negatywne jasno, pozytywne ciemno."""
  if r >= 0.75: return "█"
  if r >= 0.50: return "▓"
  if r >= 0.25: return "▒"
  if r >= 0.05: return "░"
  if r > -0.05: return "·"
  if r > -0.25: return "-"
  if r > -0.50: return "="
  if r > -0.75: return "#"
  return "@"

def make_california_housing(n=5000, seed=42):
  """Generuje syntetyczny dataset wzorowany na California Housing."""
  rng = np.random.default_rng(seed)
  # Współrzędne geograficzne w obszarze Kalifornii (szerokość ~32-42, długość~-124..-114)
  latitude = rng.uniform(32.5, 41.5, n)
  longitude = rng.uniform(-123.5, -114.5, n)
  # Dochód: rozkład skośny
  med_inc = rng.gamma(shape=2.0, scale=1.8, size=n) + 0.5
  # Wiek domu
  house_age = rng.integers(1, 55, n)
  # Pokoje/sypialnie/populacja
  ave_rooms = rng.normal(5.5, 1.5, n).clip(1, 15)
  ave_bedrms = (ave_rooms * rng.uniform(0.15, 0.25, n)).clip(0.5, 5)
  population = rng.integers(50, 5000, n)
  ave_occup = rng.normal(3.0, 0.8, n).clip(1, 10)
  # Cena domu (w 100_000 USD) - zależna od dochodu, wieku i lokalizacji
  med_house_val = (
  med_inc * 0.6
  + (latitude - 36) * 0.15
  - (house_age / 100)
  + rng.normal(0, 0.3, n)
  ).clip(0.15, 5.0)
  return pd.DataFrame({
    'MedInc': med_inc.round(4),
    'HouseAge': house_age,
    'AveRooms': ave_rooms.round(4),
    'AveBedrms': ave_bedrms.round(4),
    'Population': population,
    'AveOccup': ave_occup.round(4),
    'Latitude': latitude.round(4),
    'Longitude': longitude.round(4),
    'MedHouseVal': med_house_val.round(4)
    })
df_housing = make_california_housing(n=5000, seed=42)
C = np.corrcoef(df_housing.values.T)
ascii_heatmap(C, labels=df_housing.columns)

z = []

for i in range(len(C)):
  for j in range(i + 1, len(C)):
      z.append((abs(C[i][j]), i, j))

z.sort(reverse=True)

for wartosc, i, j in z[:3]:
  print(f"{df_housing.columns[i]} - {df_housing.columns[j]}: {wartosc:.3f}")
#MedInc - MedHouseVal - czym większe dochody tym na lepszy dom można sobie pozwolić
#Avg bedrom i Avg room - czym więcej pomieszczeń tym więcej sypialn bo najczęściej będą to domy wielorodzinne
#Latitude - MedHouseVal - najpewniej domy w pewnych regionach są warte więcej. Słaba korelacja


ASCII heatmap macierzy korelacji (legenda: █≥.75 ▓≥.5 ▒≥.25 ░≥.05 ·≈0 -/=≤0#/@≤-.5):
           MedInc HouseAg AveRoom AveBedr Populat AveOccu Latitud Longitu MedHous
    MedInc  █ +1.00 · +0.00 · -0.01 · -0.01 · +0.01 · +0.01 · +0.01 · +0.00 █ +0.90
   HouseAg  · +0.00 █ +1.00 · +0.00 · +0.01 · -0.01 · -0.01 · +0.00 · -0.00 - -0.09
   AveRoom  · -0.01 · +0.00 █ +1.00 █ +0.87 · +0.02 · +0.02 · +0.01 · +0.01 · -0.01
   AveBedr  · -0.01 · +0.01 █ +0.87 █ +1.00 · +0.01 · +0.02 · +0.01 · +0.02 · -0.01
   Populat  · +0.01 · -0.01 · +0.02 · +0.01 █ +1.00 · +0.04 · +0.02 · +0.02 · +0.02
   AveOccu  · +0.01 · -0.01 · +0.02 · +0.02 · +0.04 █ +1.00 · +0.02 · +0.00 · +0.02
   Latitud  · +0.01 · +0.00 · +0.01 · +0.01 · +0.02 · +0.02 █ +1.00 · +0.02 ▒ +0.25
   Longitu  · +0.00 · -0.00 · +0.01 · +0.02 · +0.02 · +0.00 · +0.02 █ +1.00 · +0.00
   MedHous  █ +0.90 - -0.09 · -0.01 · -0.01 · +0.02 · +0.02 ▒ +0.25 · +0.00 █ +1.00
MedInc - MedHouseVal: 0.903
AveRooms - AveBedrms: 0.867
Latitude - MedHouseV

In [34]:
# Zadanie 6 – Test t dwóch prób
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# Podział na dwie grupy
age_survived = df.loc[df["Survived"] == 1, "Age"].dropna().values
age_died = df.loc[df["Survived"] == 0, "Age"].dropna().values

print("Liczba osób, które przeżyły:", len(age_survived))
print("Liczba osób, które nie przeżyły:", len(age_died))

print("Średni wiek ocalałych:", round(age_survived.mean(), 2))
print("Średni wiek zmarłych:", round(age_died.mean(), 2))

def welch_ttest_manual(a, b):
  n1, n2 = len(a), len(b)

  mean1, mean2 = np.mean(a), np.mean(b)
  var1, var2 = np.var(a, ddof=1), np.var(b, ddof=1)

  # Statystyka t
  t = (mean1 - mean2) / np.sqrt(var1/n1 + var2/n2)

  # Stopnie swobody – wzór Welcha-Satterthwaite'a
  df = (var1/n1 + var2/n2)**2 / (
    (var1/n1)**2/(n1-1) +
    (var2/n2)**2/(n2-1)
  )

  return t, df

t_stat, degrees_freedom = welch_ttest_manual(
  age_survived, age_died
)

print("Statystyka t:", round(t_stat, 4))
print("Stopnie swobody:", round(degrees_freedom, 2))

def permutation_pvalue(a, b, n_perm=2000):
  observed_diff = abs(np.mean(a) - np.mean(b))

  combined = np.concatenate([a, b])
  n1 = len(a)

  count = 0

  rng = np.random.default_rng(42)

  for _ in range(n_perm):
    shuffled = rng.permutation(combined)

    group1 = shuffled[:n1]
    group2 = shuffled[n1:]

    perm_diff = abs(np.mean(group1) - np.mean(group2))

    if perm_diff >= observed_diff:
      count += 1

  p_value = count  / n_perm

  return p_value


p_value = permutation_pvalue(
  age_survived,
  age_died,
  n_perm=2000
)

print("Permutation p-value:", round(p_value, 5))
if(p_value<0.05):
  print("Istnieje zależność między wiekiem, a przeżyciem")
else:
  print("Nie istnieje zależność między wiekiem, a przeżyciem")

def cohens_d(a, b):
  n1, n2 = len(a), len(b)
  var1 = np.var(a, ddof=1)
  var2 = np.var(b, ddof=1)
  pooled_std = np.sqrt(
    ((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2)
  )
  d = (np.mean(a) - np.mean(b)) / pooled_std
  return d


d = cohens_d(age_survived, age_died)

print("Cohen's d:", round(d, 3))

if abs(d) < 0.2:
  print("Efekt pomijalny")
elif abs(d) < 0.5:
  print("Efekt mały")
elif abs(d) < 0.8:
  print("Efekt średni")
else:
  print("Efekt duży")


def ascii_boxplot_two(a, b):
  groups = [
    ("Survived", a),
    ("Died", b)
  ]

  all_data = np.concatenate([a, b])
  x_min = np.floor(all_data.min())
  x_max = np.ceil(all_data.max())

  width = 60

  def position(value):
    return int((value - x_min) / (x_max - x_min) * (width - 1))

  print("\nASCII BOXPLOT – Wiek pasażerów")
  print(f"Skala wieku: {x_min:.0f} - {x_max:.0f}")

  for name, data in groups:
    q1, median, q3 = np.percentile(data, [25, 50, 75])
    minimum = np.min(data)
    maximum = np.max(data)

    grid = [" "] * width

    p_min = position(minimum)
    p_q1 = position(q1)
    p_med = position(median)
    p_q3 = position(q3)
    p_max = position(maximum)

    # Wąsy
    for i in range(p_min, p_q1):
      grid[i] = "-"

    for i in range(p_q3 + 1, p_max + 1):
      grid[i] = "-"

    # Pudełko
    for i in range(p_q1, p_q3 + 1):
      grid[i] = "="

      grid[p_min] = "|"
      grid[p_max] = "|"
      grid[p_q1] = "["
      grid[p_q3] = "]"
      grid[p_med] = "|"

    print(f"{name:10} |{''.join(grid)}|")
    print(
      f"  Min={minimum:.1f}, Q1={q1:.1f}, "
      f"Mediana={median:.1f}, Q3={q3:.1f}, "
      f"Max={maximum:.1f}"
      )

ascii_boxplot_two(age_survived, age_died)


Liczba osób, które przeżyły: 290
Liczba osób, które nie przeżyły: 424
Średni wiek ocalałych: 28.34
Średni wiek zmarłych: 30.63
Statystyka t: -2.046
Stopnie swobody: 598.84
Permutation p-value: 0.04
Istnieje zależność między wiekiem, a przeżyciem
Cohen's d: -0.157
Efekt pomijalny

ASCII BOXPLOT – Wiek pasażerów
Skala wieku: 0 - 80
Survived   ||-------------[=====|=====]--------------------------------||
  Min=0.4, Q1=19.0, Mediana=28.0, Q3=36.0, Max=80.0
Died       ||--------------[====|=======]-------------------------|     |
  Min=1.0, Q1=21.0, Mediana=28.0, Q3=39.0, Max=74.0


In [38]:
import pandas as pd
import numpy as np
# Wczytanie datasetu Titanic (pd.read_csv z URL wprowadzony w L11)
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
titanic = pd.read_csv(url)
print("Dataset Titanic:")
print(titanic.head())
# Pytanie: Czy przeżycie zależy od klasy podróży?
# H₀: Przeżycie i Pclass są niezależne
# H₁: Przeżycie i Pclass są zależne
# --- Tabela kontyngencji ---
contingency_table = pd.crosstab(titanic["Pclass"], titanic["Survived"], margins=True)
print("\nTabela kontyngencji Pclass × Survived:")
print(contingency_table)
contingency_pct = pd.crosstab(titanic["Pclass"], titanic["Survived"], normalize="index") * 100
print("\nProcent przeżycia według Pclass:")
print(contingency_pct.round(2))
# --- Ręczna statystyka chi-kwadrat ---
# Chi² = Σ (O - E)² / E, gdzie E_ij = (row_total_i * col_total_j) / n
observed = pd.crosstab(titanic["Pclass"], titanic["Survived"]).values
row_totals = observed.sum(axis=1, keepdims=True)
col_totals = observed.sum(axis=0, keepdims=True)
n_total = observed.sum()
expected = row_totals @ col_totals / n_total
chi2_stat = ((observed - expected) ** 2 / expected).sum()
dof = (observed.shape[0] - 1) * (observed.shape[1] - 1)
print("\nWartości obserwowane O:")
print(observed)
print("\nWartości oczekiwane E (pod H₀ niezależności):")
print(expected.round(2))
print(f"\nChi² = Σ(O-E)²/E = {chi2_stat:.4f}")
print(f"Stopnie swobody: df = (r-1)(c-1) = {dof}")
# --- p-value przez symulację Monte Carlo ---
# Generujemy N tabel pod H₀ (niezależność, stałe totale) i liczymy chi² dla każdej.
# p-value = frakcja symulowanych chi² >= obserwowane chi².
def chi2_monte_carlo_pvalue(observed, n_sim=5000, seed=42):
  rng = np.random.default_rng(seed)
  row_totals = observed.sum(axis=1)
  col_totals = observed.sum(axis=0)
  n = observed.sum()
  expected = np.outer(row_totals, col_totals) / n
  def chi2(table, exp):
    return ((table - exp) ** 2 / exp).sum()
  observed_chi2 = chi2(observed, expected)
  count_extreme = 0
  for _ in range(n_sim):
    # Generujemy tabelę pod H0 przez multinomial z pooled probabilities
    probs = expected.flatten() / n
    table_sim = rng.multinomial(n, probs).reshape(observed.shape)
    exp_sim = np.outer(table_sim.sum(axis=1), table_sim.sum(axis=0)) / n
    exp_sim = np.maximum(exp_sim, 1e-9) # zabezpieczenie przed dzieleniem przez 0
    if chi2(table_sim, exp_sim) >= observed_chi2:
      count_extreme += 1
  return count_extreme / n_sim
p_value = chi2_monte_carlo_pvalue(observed, n_sim=3000, seed=42)
print("\n" + "=" * 60)
print("TEST CHI-KWADRAT (Monte Carlo, 3000 symulacji)")
print("=" * 60)
print(f"H₀: Przeżycie i Pclass są niezależne")
print(f"H₁: Przeżycie i Pclass są zależne")
print(f"Chi² obserwowane = {chi2_stat:.4f}, df = {dof}")

print(f"p-value (MC) ≈ {p_value:.6f}")
alpha = 0.05
if p_value < alpha:
  print(f"WNIOSEK: Odrzucamy H₀ (p < {alpha})")
  print("Przeżycie ZALEŻY od Pclass")
else:
  print(f"WNIOSEK: Nie odrzucamy H₀ (p >= {alpha})")
# --- Cramer's V - siła związku ---
min_dim = min(observed.shape[0] - 1, observed.shape[1] - 1)
cramers_v = np.sqrt(chi2_stat / (n_total * min_dim))
strength = "słaby" if cramers_v < 0.1 else ("średni" if cramers_v < 0.3 else "silny")
print(f"\nCramer's V = {cramers_v:.4f} → związek {strength}")
# --- ASCII wizualizacja: stacked bar chart ---
Pclass_labels = list(contingency_pct.index)
survived_labels = ["Nie przeżył (0)", "Przeżył (1)"]
def ascii_stacked_bars(pct_table, row_labels, col_labels, width=50):
  print("\nProcent przeżycia według Pclass:")
  print(f" {'':<12} {'0%':<5}{'25%':<13}{'50%':<13}{'75%':<13}{'100%':<5}")
  print(f" {'':<12} +-----+------------+------------+------------+")
  for i, rlabel in enumerate(row_labels):
    row = [" "] * width
    cum = 0
    for j, clabel in enumerate(col_labels):
      pct = pct_table.iloc[i, j]
      n_chars = int(round(pct / 100 * width))
      char = "░" if j == 0 else "█"
      for k in range(cum, min(cum + n_chars, width)):
        row[k] = char
      cum += n_chars
    print(f" {rlabel:<12} |{''.join(row)}| vals:{list(pct_table.iloc[i].round(1).values)}")
  print(f" Legenda: ░ = nie przeżył, █ = przeżył")
ascii_stacked_bars(contingency_pct, Pclass_labels, survived_labels)
# --- ASCII tabela kontyngencji ---
print("\nTabela kontyngencji (wartości obserwowane):")
print(f" {'':<12} {'Nie przeżył':>12} {'Przeżył':>10}")
for i, label in enumerate(Pclass_labels):
  print(f" {label:<12} {observed[i,0]:>12} {observed[i,1]:>10}")

Dataset Titanic:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   

In [53]:
# zadanie 9
import pandas as pd
import numpy as np
import math

# 1. Symulacja 1000 eksperymentów po 20 rzutów
rng = np.random.default_rng(seed=42)
sim = rng.binomial(n=20, p=0.5, size=1000)

# 2. Histogram symulacji
hist = np.bincount(sim, minlength=21)

# 3. Funkcja PMF rozkładu dwumianowego
def binom_pmf(k, n, p):
  return math.comb(n, k) * (p ** k) * ((1-p) ** (n-k))

n = 20
p = 0.5

x_binom = np.arange(0, n + 1)
pmf_binom = np.array([binom_pmf(int(k), n, p) for k in x_binom])

# 4. Funkcja ASCII bar chart
def ascii_barplot_pmf(xs, ps, title, bar_char="█", max_width=40):
  print(f"\n{title}")

  p_max = ps.max() if ps.max() > 0 else 1

  for xi, pi in zip(xs, ps):
    bar_len = int(pi / p_max * max_width)
    print(f"x={int(xi):>2} p={pi:.4f} {bar_char * bar_len}")

# 5. Porównanie symulacji i teorii
# Histogram zamieniamy na częstości względne
sim_pmf = hist / len(sim)

ascii_barplot_pmf(
  x_binom,
  sim_pmf,
  "Rozklad empiryczny - symulacja",
  bar_char="█"
)

ascii_barplot_pmf(
  x_binom,
  pmf_binom,
  "Rozklad teoretyczny B(20, 0.5)",
  bar_char="·"
)

# 6. Porównanie średnich
mean_empirical = sim.mean()
mean_theoretical = n * p

print("\nPorownanie srednich:")
print(f"Srednia empiryczna: {mean_empirical:.3f}")
print(f"Srednia teoretyczna: {mean_theoretical:.3f}")
print(f"Roznica: {abs(mean_empirical - mean_theoretical):.3f}")




Rozklad empiryczny - symulacja
x= 0 p=0.0000 
x= 1 p=0.0000 
x= 2 p=0.0000 
x= 3 p=0.0010 
x= 4 p=0.0060 █
x= 5 p=0.0130 ███
x= 6 p=0.0430 ██████████
x= 7 p=0.0760 █████████████████
x= 8 p=0.1240 █████████████████████████████
x= 9 p=0.1510 ███████████████████████████████████
x=10 p=0.1710 ████████████████████████████████████████
x=11 p=0.1560 ████████████████████████████████████
x=12 p=0.1330 ███████████████████████████████
x=13 p=0.0730 █████████████████
x=14 p=0.0290 ██████
x=15 p=0.0180 ████
x=16 p=0.0040 
x=17 p=0.0020 
x=18 p=0.0000 
x=19 p=0.0000 
x=20 p=0.0000 

Rozklad teoretyczny B(20, 0.5)
x= 0 p=0.0000 
x= 1 p=0.0000 
x= 2 p=0.0002 
x= 3 p=0.0011 
x= 4 p=0.0046 ·
x= 5 p=0.0148 ···
x= 6 p=0.0370 ········
x= 7 p=0.0739 ················
x= 8 p=0.1201 ···························
x= 9 p=0.1602 ····································
x=10 p=0.1762 ········································
x=11 p=0.1602 ····································
x=12 p=0.1201 ···························
x=1

In [57]:
# zadanie 16
import pandas as pd
import numpy as np
import math

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

fares = df["Fare"].dropna().values
fares = fares[fares > 0]

n = len(fares)

# Rozkład normalny
mu_norm = fares.mean()
sigma_norm = fares.std(ddof=1)

# Rozkład lognormalny
log_fares = np.log(fares)
mu_log = log_fares.mean()
sigma_log = log_fares.std(ddof=1)

# Rozkład wykładniczy
lambda_exp = 1 / fares.mean()

print("PARAMETRY ROZKLADOW:")
print(f"Normalny: mu={mu_norm:.3f}, sigma={sigma_norm:.3f}")
print(f"Lognormalny: mu_log={mu_log:.3f}, sigma_log={sigma_log:.3f}")
print(f"Wykladniczy: lambda={lambda_exp:.5f}")


def ecdf(x, data):
  sorted_data = np.sort(data)
  return np.searchsorted(sorted_data, x, side="right") / len(data)


def normal_cdf(x, mu, sigma):
  return np.array([
    0.5 * (1 + math.erf((value - mu) / (sigma * math.sqrt(2))))
    for value in np.asarray(x)
  ])


def cdf_normal(x):
  return normal_cdf(x, mu_norm, sigma_norm)


def cdf_lognormal(x):
  return normal_cdf(np.log(x), mu_log, sigma_log)


def cdf_exponential(x):
  return 1 - np.exp(-lambda_exp * x)


def ks_test_manual(data, cdf_function):
  sorted_data = np.sort(data)
  n = len(sorted_data)
  ecdf_upper = np.arange(1, n + 1) / n
  ecdf_lower = np.arange(0, n) / n
  cdf_theory = cdf_function(sorted_data)
  d_plus = np.max(ecdf_upper - cdf_theory)
  d_minus = np.max(cdf_theory - ecdf_lower)
  D = max(d_plus, d_minus)
  p_value = min(1.0, 2 * np.exp(-2 * n * D**2))
  return D, p_value


results = []

for name, cdf in [
  ("Normalny", cdf_normal),
  ("Lognormalny", cdf_lognormal),
  ("Wykladniczy", cdf_exponential)
]:
  D, p_value = ks_test_manual(fares, cdf)
  results.append((name, D, p_value))

results_df = pd.DataFrame(
  results,
  columns=["Rozklad", "Statystyka D", "p-value"]
)

print("\nWYNIKI TESTU K-S:")
print(results_df.to_string(index=False, formatters={
  "Statystyka D": "{:.5f}".format,
  "p-value": "{:.6g}".format
}))

best = min(results, key=lambda row: row[1])

print(f"\nNajlepszy rozklad: {best[0]}")
print(f"Najmniejsza statystyka D: {best[1]:.5f}")


# 10. ASCII histogram i teoretyczne PDF
def normal_pdf(x, mu, sigma):
  return (
    np.exp(-0.5 * ((x - mu) / sigma)**2)/ (sigma * np.sqrt(2 * np.pi))
  )


def lognormal_pdf(x, mu, sigma):
  return (
    np.exp(-0.5 * ((np.log(x) - mu) / sigma)**2)/ (x * sigma * np.sqrt(2 * np.pi))
  )


def exponential_pdf(x, lam):
    return lam * np.exp(-lam * x)


bins = np.linspace(0, fares.max(), 31)
hist, edges = np.histogram(fares, bins=bins)

bin_centers = (edges[:-1] + edges[1:]) / 2
bin_width = edges[1] - edges[0]

x_grid = np.linspace(0.01, fares.max(), 200)

pdf_normal = normal_pdf(x_grid, mu_norm, sigma_norm)
pdf_log = lognormal_pdf(x_grid, mu_log, sigma_log)
pdf_exp = exponential_pdf(x_grid, lambda_exp)

scale = n * bin_width

print("\nASCII HISTOGRAM I DOPASOWANE PDF:")

max_hist = hist.max()
max_pdf = max(
  pdf_normal.max(),
  pdf_log.max(),
  pdf_exp.max()
)

for i in range(len(hist)):
  center = bin_centers[i]
  hist_len = int(hist[i] / max_hist * 30)
  print(f"{center:7.1f} | {'█' * hist_len}")

print("\nPDF - NORMALNY:")
for x, y in zip(x_grid[::10], pdf_normal[::10]):
  print(f"{x:7.1f} | {'·' * int(y / max_pdf * 30)}")

print("\nPDF - LOGNORMALNY:")
for x, y in zip(x_grid[::10], pdf_log[::10]):
  print(f"{x:7.1f} | {'·' * int(y / max_pdf * 30)}")

print("\nPDF - WYKLADNICZY:")
for x, y in zip(x_grid[::10], pdf_exp[::10]):
  print(f"{x:7.1f} | {'·' * int(y / max_pdf * 30)}")


PARAMETRY ROZKLADOW:
Normalny: mu=32.756, sigma=49.937
Lognormalny: mu_log=2.943, sigma_log=0.936
Wykladniczy: lambda=0.03053

WYNIKI TESTU K-S:
    Rozklad Statystyka D     p-value
   Normalny      0.29566 6.13111e-67
Lognormalny      0.13421  3.9458e-14
Wykladniczy      0.17976 5.16713e-25

Najlepszy rozklad: Lognormalny
Najmniejsza statystyka D: 0.13421

ASCII HISTOGRAM I DOPASOWANE PDF:
    8.5 | ██████████████████████████████
   25.6 | ███████████
   42.7 | ██
   59.8 | ██
   76.8 | ██
   93.9 | 
  111.0 | 
  128.1 | 
  145.2 | 
  162.2 | 
  179.3 | 
  196.4 | 
  213.5 | 
  230.5 | 
  247.6 | 
  264.7 | 
  281.8 | 
  298.9 | 
  315.9 | 
  333.0 | 
  350.1 | 
  367.2 | 
  384.2 | 
  401.3 | 
  418.4 | 
  435.5 | 
  452.6 | 
  469.6 | 
  486.7 | 
  503.8 | 

PDF - NORMALNY:
    0.0 | ·····
   25.8 | ······
   51.5 | ······
   77.2 | ····
  103.0 | ··
  128.7 | ·
  154.5 | 
  180.2 | 
  206.0 | 
  231.7 | 
  257.5 | 
  283.2 | 
  308.9 | 
  334.7 | 
  360.4 | 
  386.2 | 
  411.9 | 
 